## **Import Libraries**
___

In [ ]:
import pandas as pd
import numpy as np
import warnings
from fredapi import Fred
warnings.filterwarnings('ignore') 
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from statsmodels.tsa.stattools import adfuller, kpss, ccf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from pathlib import Path
TEMPLATE = 'plotly_white'

NAVY  = '#1F3864'
BLUE  = '#2E75B6'
RED   = '#C0392B'
GREEN = '#27AE60'
AMBER = '#E67E22'
GREY  = '#7F8C8D'
LBLUE = '#AED6F1'

RECESSIONS_US = [
    ('1990-07-01', '1991-03-31', 'Early 1990s'),
    ('2001-03-01', '2001-12-31', 'Dot-com'),
    ('2007-12-01', '2009-06-30', 'GFC'),
    ('2020-02-01', '2020-04-30', 'COVID'),
]
RECESSIONS_UK = [
    ('1990-07-01', '1991-03-31', 'Early 1990s'),
    ('2008-01-01', '2009-06-30', 'GFC'),
    ('2020-01-01', '2020-06-30', 'COVID'),
]

def add_recessions(fig, recessions, row=None, col=None):
    """Add shaded recession bands to a plotly figure."""
    kwargs = dict(row=row, col=col) if row is not None else {}
    for start, end, label in recessions:
        fig.add_vrect(
            x0=start, x1=end,
            fillcolor='lightgrey', opacity=0.35,
            layer='below', line_width=0,
            annotation_text=label,
            annotation_position='top left',
            annotation_font_size=8,
            annotation_font_color=GREY,
            **kwargs
        )
    return fig

BASE        = Path('../Data Collection')   # Adjust as needed
MACRO_US    = BASE / 'MacroVariables_US.csv'
MACRO_UK    = BASE / 'MacroVariables_UK.csv'
DELINQ_US   = BASE / 'DelinquencyRate_US.csv'
DELINQ_UK   = BASE / 'DelinquencyRate_UK.csv'

print('Libraries loaded.')
print(f'US macro path:  {MACRO_US}')
print(f'UK macro path:  {MACRO_UK}')

Libraries loaded.
US macro path:  ..\Data Collection\MacroVariables_US.csv
UK macro path:  ..\Data Collection\MacroVariables_UK.csv


In [2]:
# from fredapi import Fred

# fred = Fred(api_key='ca04d9a1acca4616dd8c051986a1e29e ')

# delinquency_raw = fred.get_series('DRALACBS')
# delinquency_raw.index = delinquency_raw.index.to_period('Q').to_timestamp('Q')
# delinquency_rate = delinquency_raw.copy()
# delinquency_rate.name = 'delinquency_rate'

In [3]:
# print(delinquency_rate.head())
# print(delinquency_rate.shape)
# print(delinquency_rate.dtypes)

In [4]:
# import os

# # Save delinquency rate to CSV
# output_dir = r"C:/Users/andre/OneDrive/LSE/ST498 - Capstone Project/ST-498/Data Collection"
# delinquency_rate.to_csv(os.path.join(output_dir, "DelinquencyRate_US.csv"), header=True)

# **Exploratory Data Analysis — Probability of Default (PD) Model**
___

This notebook loads the target variable and macroeconomic predictor variables to be used in the PD model. 

- **Target variable:** Delinquency Rate on All Loans and Leases (DRALACBS), sourced from the Federal Reserve Bank of St. Louis (FRED). This series serves as a proxy for probability of default at the aggregate level.
- **Macroeconomic variables:** Quarterly macroeconomic indicators for the UK and US, compiled in `MacroVariables_UK.csv` and `MacroVariables_US.csv`.

## **1: Sample Window Definition & Target Variable Integration**
___

IFRS 9 requires ECL models to reflect past events, current conditions, and forecasts of future economic conditions (Bank & Eder 2021, p. 4). Reliable macro-PD coefficient estimation requires coverage of multiple full credit cycles. The 1990 start date covers four stress episodes and is the natural completeness breakpoint in both datasets.

In [5]:
# Switch between US and UK datasets by changing this variable:
COUNTRY = 'US'   # switch to 'UK' to run on UK data

if COUNTRY == 'US':
    MACRO_PATH  = MACRO_US
    DELINQ_PATH = DELINQ_US
    PREFIX      = 'us_'
    RECESSIONS  = RECESSIONS_US
elif COUNTRY == 'UK':
    MACRO_PATH  = MACRO_UK
    DELINQ_PATH = DELINQ_UK
    PREFIX      = 'uk_'
    RECESSIONS  = RECESSIONS_UK

WINDOW_START = '1990-01-01'
WINDOW_END   = '2024-12-31'
TARGET       = 'delinquency_rate'

print(f'{COUNTRY} dataset selected')

# 1.1  Load and trim macro data
macro_raw = pd.read_csv(MACRO_PATH, index_col=0, parse_dates=True)
macro_raw.index = (pd.DatetimeIndex(macro_raw.index)
                   .to_period('Q').to_timestamp('Q'))
macro_raw = macro_raw.sort_index()
df = macro_raw.loc[WINDOW_START:WINDOW_END].copy()

print(f'\nFull dataset:      {macro_raw.shape[0]} quarters '
      f'({macro_raw.index.min().date()} to {macro_raw.index.max().date()})')
print(f'Analytical window: {df.shape[0]} quarters '
      f'({df.index.min().date()} to {df.index.max().date()})')

# 1.2  Load delinquency rate
delinq_raw = pd.read_csv(DELINQ_PATH, index_col=0, parse_dates=True)
delinq_raw.index = (pd.DatetimeIndex(delinq_raw.index)
                    .to_period('Q').to_timestamp('Q'))
delinq_raw = delinq_raw.sort_index()
delinq_raw.columns = [TARGET]
delinq = delinq_raw.loc[WINDOW_START:WINDOW_END, TARGET]

print(f'Delinquency rate (target):  {len(delinq)} quarters  '
      f'Min {delinq.min():.2f}%  '
      f'Max {delinq.max():.2f}%  '
      f'Mean {delinq.mean():.2f}%')

# 1.3  Merge 
df = df.join(delinq, how='left')
print(f'\nMerged shape: {df.shape}  |  Target NaN: {df[TARGET].isna().sum()}\n')

# 1.4  Completeness report
completeness = pd.DataFrame({
    'Non-null':    df.notna().sum(),
    'Missing':     df.isna().sum(),
    'Missing (%)': (df.isna().sum() / len(df) * 100).round(1),
    'First valid': df.apply(lambda s: s.first_valid_index()
                            .strftime('%Y-Q') if s.first_valid_index() else None),
    'Last valid':  df.apply(lambda s: s.last_valid_index()
                            .strftime('%Y-Q') if s.last_valid_index() else None),
})

def colour_missing(val):
    if val == 0:   return 'background-color: #D5F5E3'
    if val <= 5:   return 'background-color: #FEF9E7'
    if val <= 15:  return 'background-color: #FDEBD0'
    return 'background-color: #FADBD8'

display(completeness.style
        .map(colour_missing, subset=['Missing (%)'])
        .set_caption(f'Table 1.1 — Data Completeness Report '
                     f'({COUNTRY}, 1990–2024)'))

# 1.5  Target variable — Deliquency rate time series with recession bands and stress peaks
mean_val = df[TARGET].mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df[TARGET],
    mode='lines',
    name='Delinquency Rate',
    line=dict(color=NAVY, width=2.5),
    fill='tozeroy',
    fillcolor='rgba(46,117,182,0.10)',
    hovertemplate='%{x|%Y Q%q}<br>Rate: %{y:.2f}%<extra></extra>',
))

fig.add_hline(
    y=mean_val,
    line_dash='dash', line_color=GREY, line_width=1.5,
    annotation_text=f'Mean = {mean_val:.2f}%',
    annotation_position='top right',
    annotation_font_color=GREY,
)

# Recession bands
fig = add_recessions(fig, RECESSIONS)

# Annotate stress peaks
peak_windows = [('2007', '2010'), ('2019', '2021')]
for start, end in peak_windows:
    subset = df[TARGET].loc[start:end].dropna()
    if not subset.empty:
        peak_date = subset.idxmax()
        peak_val  = subset.max()
        fig.add_annotation(
            x=peak_date,
            y=peak_val,
            text=f'{peak_val:.1f}%',
            showarrow=True,
            arrowhead=2,
            arrowcolor=RED,
            font=dict(color=RED, size=11, family='Arial'),
            bgcolor='white',
            bordercolor=RED,
            borderwidth=1,
            ay=-45,
        )

fig.update_layout(
    title=dict(
        text=f'Figure 1.1 — Delinquency Rate, All Commercial Banks '
             f'({COUNTRY}, 1990–2024)',
        font=dict(size=14, color=NAVY, family='Arial'),
    ),
    xaxis=dict(title='', showgrid=True, gridcolor='#E0E0E0'),
    yaxis=dict(title='Delinquency Rate (%)',
               showgrid=True, gridcolor='#E0E0E0'),
    template=TEMPLATE,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='right', x=1),
    height=430,
    margin=dict(t=80, b=40, l=60, r=40),
)
fig.show()

# 1.6  Raw variables overview
raw_vars = [c for c in df.columns if c != TARGET]
ncols    = 4
nrows    = int(np.ceil(len(raw_vars) / ncols))

fig2 = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[v.replace(PREFIX, '').replace('_', ' ').title()
                    for v in raw_vars],
    vertical_spacing=0.08,
    horizontal_spacing=0.06,
)

for i, var in enumerate(raw_vars):
    row = i // ncols + 1
    col = i %  ncols + 1
    fig2.add_trace(
        go.Scatter(
            x=df.index, y=df[var],
            mode='lines',
            name=var,
            line=dict(color=BLUE, width=1.4),
            hovertemplate=var + '<br>%{x|%Y}<br>%{y:.2f}<extra></extra>',
            showlegend=False,
        ),
        row=row, col=col,
    )
    # Add recession bands per subplot
    for start, end, label in RECESSIONS:
        fig2.add_vrect(
            x0=start, x1=end,
            fillcolor='lightgrey', opacity=0.3,
            layer='below', line_width=0,
            row=row, col=col,
        )

fig2.update_layout(
    title=dict(
        text=f'Figure 1.2 — All Raw Variables ({COUNTRY}, 1990–2024  |  '
             f'grey = recessions)',
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    template=TEMPLATE,
    height=nrows * 220,
    margin=dict(t=80, b=40, l=40, r=40),
)
fig2.show()

# 1.7  Consumer confidence diagnostic
cc_col = PREFIX + 'consumer_confidence'
if cc_col in df.columns:
    cc = df[cc_col].dropna()
    print(f'\n=== Consumer Confidence Diagnostic ({COUNTRY}) ===')
    print(f'  Observations : {len(cc)}')
    print(f'  Range        : {cc.min():.3f}  to  {cc.max():.3f}')
    print(f'  Std          : {cc.std():.3f}')
    print(f'  Note         : Expected range ~50–110 for US Michigan Sentiment.')
    if cc.std() < 5:
        print(f'  FLAG         : Std of {cc.std():.2f} is implausibly narrow.')

US dataset selected

Full dataset:      428 quarters (1919-03-31 to 2025-12-31)
Analytical window: 140 quarters (1990-03-31 to 2024-12-31)
Delinquency rate (target):  140 quarters  Min 1.19%  Max 7.40%  Mean 2.95%

Merged shape: (140, 17)  |  Target NaN: 0



,Non-null,Missing,Missing (%),First valid,Last valid
us_house_price_idx,140,0,0.000000,1990-Q,2024-Q
us_cpi,140,0,0.000000,1990-Q,2024-Q
us_unemployment,140,0,0.000000,1990-Q,2024-Q
us_consumer_confidence,140,0,0.000000,1990-Q,2024-Q
us_real_gdp,140,0,0.000000,1990-Q,2024-Q
us_gdp_yoy_growth,140,0,0.000000,1990-Q,2024-Q
us_bond_yield_10y,140,0,0.000000,1990-Q,2024-Q
us_reer,124,16,11.400000,1994-Q,2024-Q
us_oil_price,140,0,0.000000,1990-Q,2024-Q
us_credit,140,0,0.000000,1990-Q,2024-Q



=== Consumer Confidence Diagnostic (US) ===
  Observations : 140
  Range        : 94.780  to  102.192
  Std          : 1.248
  Note         : Expected range ~50–110 for US Michigan Sentiment.
  FLAG         : Std of 1.25 is implausibly narrow.


### Data Quality Flag — Consumer Confidence (`us_consumer_confidence`)

**Finding:** The `us_consumer_confidence` variable shows a standard deviation of **1.25** 
across 140 quarters (1990–2024), with values ranging only from 94.78 to 102.19.

**Problem:** The University of Michigan Consumer Sentiment Index — the standard US 
consumer confidence measure — ranges approximately 50 to 110 over this period, with a 
standard deviation well above 10. A std of 1.25 is statistically inconsistent with any 
published consumer confidence series for the US.

**Likely cause:** The series appears to have been indexed or rebased to 100 using a 
specific base period during data collection, compressing the full time series into a 
narrow band around 100 and eliminating virtually all its informational content.

**Should we exclude it from the feature set?**

Two justifications:
1. **Data quality** — the series does not match the published statistical properties of 
   any standard US consumer confidence measure. The normalisation error cannot be 
   corrected without access to the original source metadata.
2. **Redundancy** — even if correctly measured, consumer confidence is largely captured 
   by the combination of unemployment, GDP growth, and equity returns already present 
   in the feature set. It does not constitute an independent signal.

**Reference:** This decision is consistent with Chawla et al. (2016), cited in 
Bank & Eder (2021, Section 11.1, p. 36), who conclude that the majority of qualitative 
indicators lag changes in the risk of default and therefore serve at best as benchmarks 
for quantitative predictors already in the model.

In [6]:
fred = Fred(api_key='ca04d9a1acca4616dd8c051986a1e29e')

# Download Fred Composite Consumer Confidence for US
cc_raw = fred.get_series('USACSCICP02STSAM')
cc_raw.index = cc_raw.index.to_period('M').to_timestamp('M')
cc_raw.name  = 'us_consumer_confidence'

print('Downloaded series:')
print(f'  Frequency : Monthly')
print(f'  Range     : {cc_raw.index.min().date()} to {cc_raw.index.max().date()}')
print(f'  Min       : {cc_raw.min():.2f}')
print(f'  Max       : {cc_raw.max():.2f}')
print(f'  Std       : {cc_raw.std():.2f}')
print(f'  NaN count : {cc_raw.isna().sum()}')

# Resample to quarterly (mean of 3 monthly obs per quarter)
cc_quarterly = (cc_raw
                .resample('QE')
                .mean()
                .rename('us_consumer_confidence'))
cc_quarterly.index = (cc_quarterly.index
                      .to_period('Q')
                      .to_timestamp('Q'))

# Trim to analytical window
cc_quarterly = cc_quarterly.loc[WINDOW_START:WINDOW_END]

print('Resampled to quarterly:')
print(f'  Observations : {len(cc_quarterly)}')
print(f'  Range        : {cc_quarterly.index.min().date()} '
      f'to {cc_quarterly.index.max().date()}')
print(f'  Min          : {cc_quarterly.min():.2f}')
print(f'  Max          : {cc_quarterly.max():.2f}')
print(f'  Mean         : {cc_quarterly.mean():.2f}')
print(f'  Std          : {cc_quarterly.std():.2f}')
print(f'  NaN count    : {cc_quarterly.isna().sum()}')

# Replace in df
df['us_consumer_confidence'] = cc_quarterly

print(f'Replaced us_consumer_confidence in df.')
print(f'NaN count after replacement: '
      f'{df["us_consumer_confidence"].isna().sum()}')

# Save as v2 to avoid overwriting original
MACRO_US_V2 = BASE / 'MacroVariables_US_v2.csv'

df.drop(columns=[TARGET]).to_csv(MACRO_US_V2)

# Update MACRO_PATH to point to v2
MACRO_US    = MACRO_US_V2
MACRO_PATH  = MACRO_US_V2

print(f'All future steps will read from MacroVariables_US_v2.csv')

# Re run the consumer confidence diagnostic on the new quarterly series
# Step 1 style analysis on the new consumer confidence series

# Descriptive stats
cc = df['us_consumer_confidence'].dropna()
print(f'\n=== Consumer Confidence Diagnostic (USACSCICP02STSAM) ===')
print(f'  Observations : {len(cc)}')
print(f'  Min          : {cc.min():.2f}')
print(f'  Max          : {cc.max():.2f}')
print(f'  Mean         : {cc.mean():.2f}')
print(f'  Std          : {cc.std():.2f}')
print(f'  Range        : {cc.max() - cc.min():.2f}')
if cc.std() > 5:
    print(f'  STATUS       : Valid range and variability confirmed.')
else:
    print(f'  STATUS       : Still implausibly narrow — check download.')

# Time series plot with recession bands and trough annotations
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['us_consumer_confidence'],
    mode='lines',
    name='Consumer Confidence',
    line=dict(color=NAVY, width=2.5),
    fill='tozeroy',
    fillcolor='rgba(46,117,182,0.10)',
    hovertemplate='%{x|%Y}<br>Index: %{y:.2f}<extra></extra>',
))

# Reference line at 100 (above = optimism, below = pessimism)
fig.add_hline(
    y=100,
    line_dash='dash', line_color=RED, line_width=1.5,
    annotation_text='100 — Neutral threshold',
    annotation_position='top right',
    annotation_font_color=RED,
)

# Mean line
fig.add_hline(
    y=cc.mean(),
    line_dash='dot', line_color=GREY, line_width=1.5,
    annotation_text=f'Mean = {cc.mean():.2f}',
    annotation_position='bottom right',
    annotation_font_color=GREY,
)

# Recession bands
fig = add_recessions(fig, RECESSIONS)

# Annotate troughs
trough_windows = [('2008', '2009'), ('2020', '2020'), ('2022', '2023')]
for start, end in trough_windows:
    subset = df['us_consumer_confidence'].loc[start:end].dropna()
    if not subset.empty:
        trough_date = subset.idxmin()
        trough_val  = subset.min()
        fig.add_annotation(
            x=trough_date,
            y=trough_val,
            text=f'{trough_val:.1f}',
            showarrow=True,
            arrowhead=2,
            arrowcolor=RED,
            font=dict(color=RED, size=10, family='Arial'),
            bgcolor='white',
            bordercolor=RED,
            borderwidth=1,
            ay=40,
        )

fig.update_layout(
    title=dict(
        text='Figure 1.3 — Fred Composite Consumer Confidence Index, US (1990–2024)<br>'
             '<sup>USACSCICP02STSAM | Quarterly mean of monthly observations | '
             'Above 100 = optimism, below 100 = pessimism</sup>',
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    xaxis=dict(title='', showgrid=True, gridcolor='#E0E0E0'),
    yaxis=dict(title='Consumer Confidence Index',
               showgrid=True, gridcolor='#E0E0E0'),
    template=TEMPLATE,
    hovermode='x unified',
    height=430,
    margin=dict(t=100, b=40, l=60, r=40),
)
fig.show()

Downloaded series:
  Frequency : Monthly
  Range     : 1960-01-31 to 2026-02-28
  Min       : 53.80
  Max       : 120.51
  Std       : 14.12
  NaN count : 0
Resampled to quarterly:
  Observations : 140
  Range        : 1990-03-31 to 2024-12-31
  Min          : 60.36
  Max          : 118.50
  Mean         : 91.69
  Std          : 14.06
  NaN count    : 0
Replaced us_consumer_confidence in df.
NaN count after replacement: 0
All future steps will read from MacroVariables_US_v2.csv

=== Consumer Confidence Diagnostic (USACSCICP02STSAM) ===
  Observations : 140
  Min          : 60.36
  Max          : 118.50
  Mean         : 91.69
  Std          : 14.06
  Range        : 58.14
  STATUS       : Valid range and variability confirmed.


## **2: Variable Classification & Redundancy Decisions**
___

We seek to prove which macreconomic indicators are most influential in predicting default risk. Before influence can be assessed, every variable must be assigned a functional group and a modelling decision made. This step eliminates redundant level/growth-rate pairs, flags variables needing transformation, and produces the cleaned feature set that all subsequent steps operate on. Follows the variable selection principle consistent with Bellini (2019), Chapter 9, cited throughout Bank & Eder (2021).

In [ ]:
# 2.1  Variable register
var_register = pd.DataFrame([
    (PREFIX+'house_price_idx',      'B — Non-stationary level',
     'Compute YoY growth',          'Transform',
     'Trending upward; no growth companion in dataset'),

    (PREFIX+'cpi',                  'A — Stationary candidate',
     'Use as-is',                   'Retain',
     'Inflation rate; mean-reverting'),

    (PREFIX+'unemployment',         'A — Stationary candidate',
     'Use as-is',                   'Retain',
     'Cyclical rate; mean-reverting — confirm in Step 3'),

    (PREFIX+'consumer_confidence',  'A — Stationary candidate',
     'Use as-is',                   'Retain',
     'Replaced with OECD USACSCICP02STSAM. Std > 10, range 58–117. '
     'Mean-reverting around 100 by construction.'),

    (PREFIX+'real_gdp',             'C — Redundant pair',
     'Exclude; use YoY growth rate',   'Exclude',
     'Level is I(1); gdp_yoy_growth is the stationary companion'),

    (PREFIX+'gdp_yoy_growth',       'A — Stationary candidate',
     'Use as-is',                   'Retain',
     'Year-on-year growth rate; expected stationary'),

    (PREFIX+'bond_yield_10y',       'A — Stationary candidate',
     'Use as-is; test carefully',   'Retain',
     'Interest rate; near-unit-root behaviour common — confirm in Step 3'),

    (PREFIX+'reer',                 'B — Non-stationary level',
     'Interpolate 1990–1993 gap; first diff if I(1)', 'Transform',
     '16 missing values 1990–1993; interpolate then test stationarity'),

    (PREFIX+'oil_price',            'B — Non-stationary level',
     'Compute YoY growth',          'Transform',
     'Volatile level; YoY growth stationary and interpretable'),

    (PREFIX+'credit',               'C — Redundant pair',
     'Exclude; use QoQ growth',        'Exclude',
     'Level is I(1); credit_qoq_growth is the stationary companion'),

    (PREFIX+'credit_qoq_growth',    'A — Stationary candidate',
     'Use as-is',                   'Retain',
     'Quarter-on-quarter credit growth; expected stationary'),

    (PREFIX+'credit_yoy_growth',    'C — Redundant pair',
    'Excluded; use QoQ growth',    'Exclude',
    'Smoothed version of credit_qoq_growth — '
    'QoQ preferred as frequency match for quarterly model'),

    (PREFIX+'industrial_production','B — Non-stationary level',
     'Compute YoY growth',          'Transform',
     'Index level; YoY growth stationary'),

    (PREFIX+'vix',                  'A — Stationary candidate',
     'Log transform',               'Transform',
     'Stationary but strongly right-skewed; log reduces skewness'),

    (PREFIX+'sp500_close' if COUNTRY == 'US' else PREFIX+'ftse_close',
     'C — Redundant pair',
     'Exclude; use QoQ return',        'Exclude',
     'Level is I(1); QoQ return is the stationary companion'),

    (PREFIX+'sp500_qoq_growth' if COUNTRY == 'US' else PREFIX+'ftse_qoq_growth',
     'A — Stationary candidate',
     'Use as-is',                   'Retain',
     'Quarterly equity index return; expected stationary'),

], columns=['Variable', 'Group', 'Planned Transform', 'Decision', 'Rationale'])

DECISION_COLOURS = {
    'Retain':    '#D5F5E3',
    'Exclude':      '#FADBD8',
    'Transform': '#FEF9E7',
    'Pending':   '#EBF5FB',
}

def colour_decision(val):
    colour = DECISION_COLOURS.get(val, '#FDEBD0')
    return f'background-color: {colour}'

display(var_register.style
        .map(colour_decision, subset=['Decision'])
        .set_caption(f'Table 2.1 — Variable Register ({COUNTRY})')
        .hide(axis='index'))

Variable,Group,Planned Transform,Decision,Rationale
us_house_price_idx,B — Non-stationary level,Compute YoY growth,Transform,Trending upward; no growth companion in dataset
us_cpi,A — Stationary candidate,Use as-is,Retain,Inflation rate; mean-reverting
us_unemployment,A — Stationary candidate,Use as-is,Retain,Cyclical rate; mean-reverting — confirm in Step 3
us_consumer_confidence,A — Stationary candidate,Use as-is,Retain,"Replaced with OECD USACSCICP02STSAM. Std > 10, range 58–117. Mean-reverting around 100 by construction."
us_real_gdp,C — Redundant pair,Exclude; use YoY growth rate,Exclude,Level is I(1); gdp_yoy_growth is the stationary companion
us_gdp_yoy_growth,A — Stationary candidate,Use as-is,Retain,Year-on-year growth rate; expected stationary
us_bond_yield_10y,A — Stationary candidate,Use as-is; test carefully,Retain,Interest rate; near-unit-root behaviour common — confirm in Step 3
us_reer,B — Non-stationary level,Interpolate 1990–1993 gap; first diff if I(1),Transform,16 missing values 1990–1993; interpolate then test stationarity
us_oil_price,B — Non-stationary level,Compute YoY growth,Transform,Volatile level; YoY growth stationary and interpretable
us_credit,C — Redundant pair,Exclude; use QoQ growth,Exclude,Level is I(1); credit_qoq_growth is the stationary companion


In [8]:
# 2.2  Compute derived variables
# House price YoY growth
df[PREFIX+'house_price_yoy'] = (df[PREFIX+'house_price_idx']
                                 .pct_change(4) * 100)

# Industrial production YoY growth
df[PREFIX+'indprod_yoy'] = (df[PREFIX+'industrial_production']
                             .pct_change(4) * 100)

# Oil price YoY growth
df[PREFIX+'oil_yoy'] = df[PREFIX+'oil_price'].pct_change(4) * 100

# Log VIX
vix_col = PREFIX + 'vix'
if vix_col in df.columns:
    df[PREFIX+'log_vix'] = np.log(df[vix_col])

# REER — interpolate gap then first difference
df[PREFIX+'reer_interp'] = (df[PREFIX+'reer']
                             .interpolate(method='linear',
                                          limit_direction='both'))
df[PREFIX+'reer_diff'] = df[PREFIX+'reer_interp'].diff()

derived = [
    PREFIX+'house_price_yoy',
    PREFIX+'indprod_yoy',
    PREFIX+'oil_yoy',
    PREFIX+'log_vix',
    PREFIX+'reer_diff',
]

print('Derived variables created:')
for v in derived:
    if v in df.columns:
        print(f'  {v:42s}  non-null: {df[v].notna().sum()}')

Derived variables created:
  us_house_price_yoy                          non-null: 136
  us_indprod_yoy                              non-null: 136
  us_oil_yoy                                  non-null: 136
  us_log_vix                                  non-null: 140
  us_reer_diff                                non-null: 139


In [9]:
# 2.3  Define feature lists

equity_col = (PREFIX+'sp500_qoq_growth' if COUNTRY == 'US'
              else PREFIX+'ftse_qoq_growth')

# Variables actively used in modelling
FEATURES = [
    PREFIX+'gdp_yoy_growth',
    PREFIX+'unemployment',
    PREFIX+'cpi',
    PREFIX+'consumer_confidence',
    PREFIX+'bond_yield_10y',
    PREFIX+'credit_qoq_growth',
    equity_col,
    PREFIX+'log_vix',
    PREFIX+'house_price_yoy',
    PREFIX+'indprod_yoy',
    PREFIX+'oil_yoy',
    PREFIX+'reer_diff',
]

# Variables excluded from modelling and why
# These remain in df — decisions are documented only
FEATURES_EXCLUDED = {
    PREFIX+'real_gdp':     'Level I(1) — redundant with gdp_yoy_growth',
    PREFIX+'credit':       'Level I(1) — redundant with credit_qoq_growth',
    PREFIX+'credit_yoy_growth':
                           'Smoothed version of credit_qoq_growth — '
                           'QoQ preferred as frequency match for quarterly model',
    PREFIX+'sp500_close' if COUNTRY == 'US' else PREFIX+'ftse_close':
                           'Level I(1) — redundant with equity QoQ return',
}

# Working analytical dataset — features + target, no columns dropped from df
df_model = df.dropna(subset=FEATURES + [TARGET]).copy()

print(f'Full df shape (untouched) : {df.shape}')
print(f'Analytical feature set    : {len(FEATURES)} variables')
print(f'Excluded (documented)     : {len(FEATURES_EXCLUDED)} variables')
print(f'Working dataset shape     : {df_model.shape}')
print(f'Date range                : {df_model.index.min().date()} '
      f'to {df_model.index.max().date()}')

print(f'\nActive features:')
for i, f in enumerate(FEATURES, 1):
    print(f'  {i:2d}.  {f}')

print(f'\nExcluded (still in df, not used in modelling):')
for var, reason in FEATURES_EXCLUDED.items():
    print(f'  {var:42s}  —  {reason}')

Full df shape (untouched) : (140, 23)
Analytical feature set    : 12 variables
Excluded (documented)     : 4 variables
Working dataset shape     : (136, 23)
Date range                : 1991-03-31 to 2024-12-31

Active features:
   1.  us_gdp_yoy_growth
   2.  us_unemployment
   3.  us_cpi
   4.  us_consumer_confidence
   5.  us_bond_yield_10y
   6.  us_credit_qoq_growth
   7.  us_sp500_qoq_growth
   8.  us_log_vix
   9.  us_house_price_yoy
  10.  us_indprod_yoy
  11.  us_oil_yoy
  12.  us_reer_diff

Excluded (still in df, not used in modelling):
  us_real_gdp                                 —  Level I(1) — redundant with gdp_yoy_growth
  us_credit                                   —  Level I(1) — redundant with credit_qoq_growth
  us_credit_yoy_growth                        —  Smoothed version of credit_qoq_growth — QoQ preferred as frequency match for quarterly model
  us_sp500_close                              —  Level I(1) — redundant with equity QoQ return


In [10]:
# 2.4  Decision summary bar chart
order   = ['Retain', 'Transform', 'Exclude']
summary = (var_register.groupby('Decision')['Variable']
           .count()
           .reindex(order)
           .fillna(0)
           .reset_index())
summary.columns = ['Decision', 'Count']

fig = go.Figure(go.Bar(
    x=summary['Decision'],
    y=summary['Count'],
    marker_color=[DECISION_COLOURS.get(d, AMBER) for d in summary['Decision']],
    marker_line_color=NAVY,
    marker_line_width=1,
    text=summary['Count'].astype(int),
    textposition='outside',
    hovertemplate='%{x}: %{y} variables<extra></extra>',
))
fig.update_layout(
    title=dict(
        text=f'Figure 2.1 — Variable Decision Summary ({COUNTRY})',
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    xaxis_title='Decision',
    yaxis_title='Number of Variables',
    yaxis=dict(range=[0, summary['Count'].max() + 2]),
    template=TEMPLATE,
    height=350,
    margin=dict(t=60, b=40, l=50, r=40),
)
fig.show()

In [11]:
# 2.5  Transformed feature set overview
ncols = 3
nrows = int(np.ceil(len(FEATURES) / ncols))

fig2 = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[v.replace(PREFIX, '').replace('_', ' ').title()
                    for v in FEATURES],
    vertical_spacing=0.09,
    horizontal_spacing=0.07,
)

for i, var in enumerate(FEATURES):
    row = i // ncols + 1
    col = i %  ncols + 1

    fig2.add_trace(
        go.Scatter(
            x=df_model.index,
            y=df_model[var],
            mode='lines',
            line=dict(color=BLUE, width=1.5),
            name=var.replace(PREFIX, '').replace('_', ' ').title(),
            hovertemplate=(var.replace(PREFIX, '').replace('_', ' ').title()
                           + '<br>%{x|%Y}<br>%{y:.2f}<extra></extra>'),
            showlegend=False,
        ),
        row=row, col=col,
    )

    fig2.add_hline(
        y=df_model[var].mean(),
        line_dash='dot',
        line_color=GREY,
        line_width=1,
        row=row, col=col,
    )

    for start, end, label in RECESSIONS:
        fig2.add_vrect(
            x0=start, x1=end,
            fillcolor='lightgrey', opacity=0.30,
            layer='below', line_width=0,
            row=row, col=col,
        )

fig2.update_layout(
    title=dict(
        text=(f'Figure 2.2 — Analytical Feature Set ({COUNTRY}, 1990–2024)  |  '
              f'grey = recessions  |  dotted = mean'),
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    template=TEMPLATE,
    height=nrows * 240,
    margin=dict(t=80, b=40, l=50, r=40),
)
fig2.show()

In [12]:
# 2.6  Descriptive statistics table
desc = df_model[FEATURES].describe().T.round(3)
desc.index = [i.replace(PREFIX, '').replace('_', ' ').title()
              for i in desc.index]
desc.columns = [c.title() for c in desc.columns]

display(desc.style
        .background_gradient(subset=['Std'], cmap='Blues')
        .background_gradient(subset=['Mean'], cmap='RdYlGn')
        .format('{:.3f}')
        .set_caption(f'Table 2.2 — Descriptive Statistics, '
                     f'Analytical Feature Set ({COUNTRY})'))

,Count,Mean,Std,Min,25%,50%,75%,Max
Gdp Yoy Growth,136.000,2.530,2.047,-7.399,1.799,2.656,3.485,12.386
Unemployment,136.000,5.717,1.767,3.533,4.392,5.383,6.692,13.000
Cpi,136.000,2.621,1.548,-1.623,1.765,2.580,3.209,8.636
Consumer Confidence,136.000,91.804,14.114,60.360,81.090,94.020,101.417,118.497
Bond Yield 10Y,136.000,4.101,1.839,0.680,2.520,4.025,5.538,8.280
Credit Qoq Growth,136.000,1.278,0.886,-0.938,0.717,1.243,1.964,2.994
Sp500 Qoq Growth,136.000,2.451,7.839,-22.558,-0.809,3.200,7.109,20.867
Log Vix,136.000,2.905,0.344,2.252,2.619,2.860,3.143,3.980
House Price Yoy,136.000,2.210,4.638,-12.567,0.163,3.424,4.933,13.026
Indprod Yoy,136.000,1.491,4.063,-15.282,-0.269,2.430,3.911,8.958


## **3: Stationarity Testing (ADF + KPSS)**
___

A foundational requirement for any time-series regression is that the variables entering the model are stationary. When a non-stationary variable is regressed on another non-stationary variable, the results are spurious — R² values appear high and coefficients appear significant purely because both series share a common trend, not because a true economic relationship exists. In the context of this project, spurious regressions would directly undermine the IFRS 9 requirement that ECL models reflect genuine forward-looking macro-default relationships rather than coincidental co-movement (Bank & Eder 2021, p. 4).

Two tests are applied in combination rather than relying on either alone. The Augmented Dickey-Fuller test (Dickey & Fuller 1979) tests the null hypothesis of a unit root — rejection is a stationary signal. The KPSS test (Kwiatkowski et al. 1992) tests the opposite null of stationarity — failure to reject is a stationary signal. Using both is standard practice precisely because their opposite null hypotheses make them complementary: a variable confirmed stationary by both provides strong evidence, while disagreement between them flags a case requiring further investigation rather than allowing a false conclusion from either test alone.

This dual-test approach is particularly important for variables like the 10-year bond yield, which shows a long secular decline from roughly 8% in 1990 to near zero post-GFC before rising again. That pattern makes it visually ambiguous — it could be a trend-stationary process or a unit root process with drift — and a single test would not resolve the question reliably. For such variables, the constant-plus-trend specification (regression='ct') is used rather than constant-only, consistent with the recommendation to match the test specification to the visible properties of the series.

In [13]:
# 3.1  Test functions for stationarity — ADF and KPSS with decision logic
def run_stationarity_tests(series, name, regression='c'):
    """
    ADF  — H0: unit root (non-stationary). Reject → stationary signal.
    KPSS — H0: stationary. Fail to reject → stationary signal.
    regression: 'c' = constant only, 'ct' = constant + trend
    """
    s = series.dropna()

    # ADF
    adf_stat, adf_p, adf_lags, _, _, _ = adfuller(
        s, autolag='AIC', regression=regression)

    # KPSS
    kpss_reg = 'ct' if regression == 'ct' else 'c'
    kpss_stat, kpss_p, _, _ = kpss(s, regression=kpss_reg, nlags='auto')

    # Decision logic
    adf_reject  = adf_p  < 0.05
    kpss_reject = kpss_p < 0.05

    if adf_reject and not kpss_reject:
        decision = 'STATIONARY'
    elif not adf_reject and kpss_reject:
        decision = 'NON-STATIONARY'
    elif not adf_reject and not kpss_reject:
        decision = 'INCONCLUSIVE'
    else:
        decision = 'CONFLICTING'

    return {
        'Variable':  name,
        'N':         len(s),
        'ADF stat':  round(adf_stat, 3),
        'ADF p':     round(adf_p, 3),
        'KPSS stat': round(kpss_stat, 3),
        'KPSS p':    round(kpss_p, 3),
        'Spec':      regression,
        'Decision':  decision,
    }

In [14]:
# 3.2  Run tests on all features
# Bond yield has a visible downward trend — use constant + trend specification
# All others use constant only
TREND_VARS = {PREFIX+'bond_yield_10y'}

results = []
for var in FEATURES:
    reg = 'ct' if var in TREND_VARS else 'c'
    results.append(
        run_stationarity_tests(df_model[var], var, regression=reg))

# Also test the target
results.append(
    run_stationarity_tests(df_model[TARGET], TARGET, regression='c'))

stat_df = pd.DataFrame(results)

DECISION_COLOURS_STAT = {
    'STATIONARY':     '#D5F5E3',
    'NON-STATIONARY': '#FADBD8',
    'INCONCLUSIVE':   '#FEF9E7',
    'CONFLICTING':    '#FDEBD0',
}

def colour_stat(val):
    return f'background-color: {DECISION_COLOURS_STAT.get(val, "white")}'

def colour_p(val):
    if val < 0.01:  return 'background-color: #D5F5E3'
    if val < 0.05:  return 'background-color: #FEF9E7'
    if val < 0.10:  return 'background-color: #FDEBD0'
    return 'background-color: #FADBD8'

display(stat_df.style
        .map(colour_stat,  subset=['Decision'])
        .map(colour_p,     subset=['ADF p'])
        .format({'ADF stat': '{:.3f}', 'ADF p': '{:.3f}',
                 'KPSS stat': '{:.3f}', 'KPSS p': '{:.3f}'})
        .set_caption(f'Table 3.1 — ADF + KPSS Stationarity Results ({COUNTRY})')
        .hide(axis='index'))

C:\Users\andre\AppData\Local\Temp\ipykernel_9544\3067022793.py:16: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.


C:\Users\andre\AppData\Local\Temp\ipykernel_9544\3067022793.py:16: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.


C:\Users\andre\AppData\Local\Temp\ipykernel_9544\3067022793.py:16: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.


C:\Users\andre\AppData\Local\Temp\ipykernel_9544\3067022793.py:16: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.


C:\Users\andre\AppData\Local\Temp\ipykernel_9544\3067022

Variable,N,ADF stat,ADF p,KPSS stat,KPSS p,Spec,Decision
us_gdp_yoy_growth,136,-2.677,0.078,0.217,0.100,c,INCONCLUSIVE
us_unemployment,136,-3.106,0.026,0.186,0.100,c,STATIONARY
us_cpi,136,-1.703,0.430,0.152,0.100,c,INCONCLUSIVE
us_consumer_confidence,136,-2.267,0.183,0.439,0.060,c,INCONCLUSIVE
us_bond_yield_10y,136,-2.092,0.551,0.249,0.010,ct,NON-STATIONARY
us_credit_qoq_growth,136,-2.824,0.055,0.346,0.100,c,INCONCLUSIVE
us_sp500_qoq_growth,136,-11.536,0.000,0.102,0.100,c,STATIONARY
us_log_vix,136,-3.982,0.002,0.096,0.100,c,STATIONARY
us_house_price_yoy,136,-2.751,0.066,0.227,0.100,c,INCONCLUSIVE
us_indprod_yoy,136,-3.167,0.022,0.405,0.075,c,STATIONARY


In [15]:
# 3.3  Dual p-value bar chart with decision logic
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'ADF p-values  (green = reject H₀ of unit root → stationary signal)',
        'KPSS p-values  (green = fail to reject H₀ of stationarity → stationary signal)',
    ],
    horizontal_spacing=0.12,
)

short_names = [v.replace(PREFIX, '').replace('_', ' ').title()
               for v in stat_df['Variable']]

# ADF — green if p < 0.05 (reject non-stationarity)
adf_colors = [GREEN if p < 0.05 else RED for p in stat_df['ADF p']]
fig.add_trace(
    go.Bar(
        x=stat_df['ADF p'],
        y=short_names,
        orientation='h',
        marker_color=adf_colors,
        marker_line_color='white',
        marker_line_width=0.5,
        name='ADF p-value',
        hovertemplate='%{y}<br>ADF p = %{x:.3f}<extra></extra>',
    ),
    row=1, col=1,
)
fig.add_vline(x=0.05, line_dash='dash', line_color=NAVY,
              line_width=1.5, row=1, col=1,
              annotation_text='p = 0.05',
              annotation_font_color=NAVY)

# KPSS — green if p > 0.05 (fail to reject stationarity)
kpss_colors = [GREEN if p > 0.05 else RED for p in stat_df['KPSS p']]
fig.add_trace(
    go.Bar(
        x=stat_df['KPSS p'],
        y=short_names,
        orientation='h',
        marker_color=kpss_colors,
        marker_line_color='white',
        marker_line_width=0.5,
        name='KPSS p-value',
        hovertemplate='%{y}<br>KPSS p = %{x:.3f}<extra></extra>',
    ),
    row=1, col=2,
)
fig.add_vline(x=0.05, line_dash='dash', line_color=NAVY,
              line_width=1.5, row=1, col=2,
              annotation_text='p = 0.05',
              annotation_font_color=NAVY)

fig.update_layout(
    title=dict(
        text=f'Figure 3.1 — Stationarity Test p-values ({COUNTRY})',
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    template=TEMPLATE,
    height=500,
    showlegend=False,
    margin=dict(t=100, b=40, l=160, r=40),
)
fig.update_xaxes(title_text='p-value', range=[0, 1])
fig.show()

In [16]:
# 3.4  Decision summary
decision_counts = (stat_df.groupby('Decision')['Variable']
                   .count().reset_index())
decision_counts.columns = ['Decision', 'Count']

fig2 = go.Figure(go.Bar(
    x=decision_counts['Decision'],
    y=decision_counts['Count'],
    marker_color=[DECISION_COLOURS_STAT.get(d, AMBER)
                  for d in decision_counts['Decision']],
    marker_line_color=NAVY,
    marker_line_width=1,
    text=decision_counts['Count'],
    textposition='outside',
    hovertemplate='%{x}: %{y} variables<extra></extra>',
))
fig2.update_layout(
    title=dict(
        text=f'Figure 3.2 — Stationarity Decision Summary ({COUNTRY})',
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    xaxis_title='Decision',
    yaxis_title='Number of Variables',
    yaxis=dict(range=[0, decision_counts['Count'].max() + 2]),
    template=TEMPLATE,
    height=350,
    margin=dict(t=60, b=40, l=50, r=40),
)
fig2.show()

In [17]:
# 3.5  Handle non-stationary / inconclusive variables

non_stat    = stat_df[stat_df['Decision'] == 'NON-STATIONARY']['Variable'].tolist()
inconclusive = stat_df[stat_df['Decision'] == 'INCONCLUSIVE']['Variable'].tolist()

FEATURES_STAT = FEATURES.copy()

# First difference NON-STATIONARY variables
if non_stat:
    print('Non-stationary — applying first difference:')
    retest = []
    for var in non_stat:
        if var in FEATURES_STAT:
            new_name = var + '_d1'
            df_model[new_name] = df_model[var].diff()
            FEATURES_STAT[FEATURES_STAT.index(var)] = new_name
            retest.append(
                run_stationarity_tests(
                    df_model[new_name].dropna(), new_name))
            print(f'  {var}  →  {new_name}')

    retest_df = pd.DataFrame(retest)
    display(retest_df.style
            .map(colour_stat, subset=['Decision'])
            .format({'ADF p': '{:.3f}', 'KPSS p': '{:.3f}'})
            .set_caption('Table 3.2 — Retest After First Differencing')
            .hide(axis='index'))

# Document inconclusive variables — retained with justification
INCONCLUSIVE_RATIONALE = {
    PREFIX+'gdp_yoy_growth':
        'ADF p=0.078 (borderline), KPSS p=0.100 (comfortable). '
        'Retained — KPSS confirms stationarity. Growth rates are '
        'mean-reverting by economic definition.',

    PREFIX+'cpi':
        'ADF p=0.430 (weak), KPSS p=0.100 (comfortable). '
        'Retained — post-COVID inflation surge (2021–2023) inflates '
        'ADF p-value by introducing a temporary trend. '
        'Inflation is mean-reverting over a full cycle.',

    PREFIX+'consumer_confidence':
        'ADF p=0.183 (borderline), KPSS p=0.060 (borderline). '
        'Retained — bounded series by construction (cannot diverge '
        'to zero or infinity), making a unit root economically '
        'implausible. Long decline since 2000 peak drives borderline result.',

    PREFIX+'credit_qoq_growth':
        'ADF p=0.055 (marginally above 5%), KPSS p=0.100 (comfortable). '
        'Retained — essentially stationary, borderline ADF driven '
        'by GFC contraction outlier.',

    PREFIX+'house_price_yoy':
        'ADF p=0.066 (marginally above 5%), KPSS p=0.100 (comfortable). '
        'Retained — KPSS confirms stationarity. YoY growth '
        'transformation successfully removes the level trend.',

    TARGET:
        'ADF p=0.234 (weak), KPSS p=0.100 (comfortable). '
        'Retained — GFC and COVID spikes inflate ADF p-value. '
        'Series is clearly mean-reverting around 2.57% over the full cycle. '
        'KPSS confirmation is the deciding criterion.',
}

print('\nInconclusive variables — retained with documented rationale:')
print('-' * 75)
for var, rationale in INCONCLUSIVE_RATIONALE.items():
    if var in inconclusive + [TARGET]:
        print(f'\n  {var}')
        print(f'  {rationale}')

# Drop NaN rows introduced by differencing
df_model = df_model.dropna(subset=FEATURES_STAT)

print(f'\n{"="*55}')
print(f'FEATURES_STAT ({len(FEATURES_STAT)} variables):')
for i, v in enumerate(FEATURES_STAT, 1):
    marker = ' ← first differenced' if v.endswith('_d1') else ''
    print(f'  {i:2d}.  {v}{marker}')
print(f'\nWorking dataset: {df_model.shape}')

Non-stationary — applying first difference:
  us_bond_yield_10y  →  us_bond_yield_10y_d1


C:\Users\andre\AppData\Local\Temp\ipykernel_9544\3067022793.py:16: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.




Variable,N,ADF stat,ADF p,KPSS stat,KPSS p,Spec,Decision
us_bond_yield_10y_d1,135,-9.020000,0.000,0.213000,0.100,c,STATIONARY



Inconclusive variables — retained with documented rationale:
---------------------------------------------------------------------------

  us_gdp_yoy_growth
  ADF p=0.078 (borderline), KPSS p=0.100 (comfortable). Retained — KPSS confirms stationarity. Growth rates are mean-reverting by economic definition.

  us_cpi
  ADF p=0.430 (weak), KPSS p=0.100 (comfortable). Retained — post-COVID inflation surge (2021–2023) inflates ADF p-value by introducing a temporary trend. Inflation is mean-reverting over a full cycle.

  us_consumer_confidence
  ADF p=0.183 (borderline), KPSS p=0.060 (borderline). Retained — bounded series by construction (cannot diverge to zero or infinity), making a unit root economically implausible. Long decline since 2000 peak drives borderline result.

  us_credit_qoq_growth
  ADF p=0.055 (marginally above 5%), KPSS p=0.100 (comfortable). Retained — essentially stationary, borderline ADF driven by GFC contraction outlier.

  us_house_price_yoy
  ADF p=0.066 (margin

### Step 3 — Stationarity Testing Results & Decisions

**Confirmed Stationary**

| Variable | ADF p | KPSS p | Note |
|----------|-------|--------|------|
| us_unemployment | 0.026 | 0.100 | Clean on both tests |
| us_sp500_qoq_growth | 0.000 | 0.100 | Clean on both tests |
| us_log_vix | 0.002 | 0.100 | Clean on both tests |
| us_indprod_yoy | 0.022 | 0.075 | Clean on both tests |
| us_oil_yoy | 0.000 | 0.100 | Clean on both tests |
| us_reer_diff | 0.000 | 0.100 | Clean on both tests |

**First Differenced**

| Variable | Original Decision | Action | Post-Differencing |
|----------|------------------|--------|-------------------|
| us_bond_yield_10y | NON-STATIONARY (ADF p=0.551, KPSS p=0.010) | First difference → us_bond_yield_10y_d1 | STATIONARY (ADF p=0.000, KPSS p=0.100) |

The secular decline in the 10-year yield from ~8% in 1990 to near zero post-GFC constitutes a genuine trend that both tests confirmed as non-stationary. First differencing produces the quarterly change in yield, which is stationary and economically more interpretable — it captures the direction of monetary policy movement rather than the absolute rate level.

**Retained with Documented Justification (Inconclusive)**

| Variable | ADF p | KPSS p | Justification |
|----------|-------|--------|---------------|
| us_gdp_yoy_growth | 0.078 | 0.100 | KPSS confirms stationarity. Growth rates are mean-reverting by economic definition. |
| us_cpi | 0.430 | 0.100 | Post-COVID inflation surge (2021–2023) introduces a temporary trend inflating ADF p-value. Inflation is mean-reverting over a full cycle. |
| us_consumer_confidence | 0.183 | 0.060 | Bounded series by construction — cannot diverge to zero or infinity, making a unit root economically implausible. Long decline since 2000 peak drives the borderline result. |
| us_credit_qoq_growth | 0.055 | 0.100 | Marginally above 5% threshold. KPSS comfortable. Borderline ADF driven by GFC contraction outlier. |
| us_house_price_yoy | 0.066 | 0.100 | KPSS confirms stationarity. YoY growth transformation successfully removes the level trend. |
| delinquency_rate (target) | 0.234 | 0.100 | GFC and COVID spikes inflate ADF p-value. Series is clearly mean-reverting around 2.57% over the full cycle. KPSS confirmation is the deciding criterion. |

**Final Feature Set After Step 3**

12 variables enter next steps via FEATURES_STAT — one variable transformed (bond yield → first difference), all others retained at their Step 2 specification.

**Working dataset: 135 rows × 24 columns — no columns removed at any point.**

Column breakdown:
- 16 original variables from MacroVariables_US_v2.csv (us_house_price_idx, us_cpi, us_unemployment, us_consumer_confidence, us_real_gdp, us_gdp_yoy_growth, us_bond_yield_10y, us_reer, us_oil_price, us_credit, us_credit_qoq_growth, us_credit_yoy_growth, us_industrial_production, us_vix, us_sp500_close, us_sp500_qoq_growth)
- 1 target variable (delinquency_rate)
- 5 derived variables from Step 2 (us_house_price_yoy, us_indprod_yoy, us_oil_yoy, us_log_vix, us_reer_diff)
- 1 intermediate variable from Step 2 (us_reer_interp)
- 1 first-differenced variable from Step 3 (us_bond_yield_10y_d1)

FEATURES_STAT acts as the analytical lens — it selects which 12 variables enter the model pipeline. All 24 columns remain accessible throughout for reference, validation, and documentation purposes.

In [18]:
# 3.6  Visualise bond yield level vs first difference
# Only bond yield required action — plot level vs d1 to document the decision

bond_var    = PREFIX + 'bond_yield_10y'
bond_d1_var = PREFIX + 'bond_yield_10y_d1'

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Bond Yield 10Y — Level  (NON-STATIONARY)',
        'Bond Yield 10Y — First Difference  (STATIONARY)',
    ],
    horizontal_spacing=0.10,
)

# Level
fig3.add_trace(
    go.Scatter(
        x=df_model.index,
        y=df_model[bond_var],
        mode='lines',
        line=dict(color=RED, width=1.8),
        name='Level',
        hovertemplate='%{x|%Y}<br>Yield: %{y:.2f}%<extra></extra>',
        showlegend=False,
    ),
    row=1, col=1,
)

# First difference
fig3.add_trace(
    go.Scatter(
        x=df_model.index,
        y=df_model[bond_d1_var],
        mode='lines',
        line=dict(color=GREEN, width=1.8),
        name='First Difference',
        hovertemplate='%{x|%Y}<br>Change: %{y:.2f}%<extra></extra>',
        showlegend=False,
    ),
    row=1, col=2,
)

# Zero line on first difference panel
fig3.add_hline(
    y=0,
    line_dash='dot', line_color=GREY, line_width=1,
    row=1, col=2,
)

# Mean line on level panel
fig3.add_hline(
    y=df_model[bond_var].mean(),
    line_dash='dot', line_color=GREY, line_width=1,
    annotation_text=f'Mean = {df_model[bond_var].mean():.2f}%',
    annotation_font_color=GREY,
    row=1, col=1,
)

# Recession bands on both panels
for start, end, label in RECESSIONS:
    for c in [1, 2]:
        fig3.add_vrect(
            x0=start, x1=end,
            fillcolor='lightgrey', opacity=0.30,
            layer='below', line_width=0,
            row=1, col=c,
        )

# ADF / KPSS annotations
fig3.add_annotation(
    x=0.25, y=0.05,
    xref='paper', yref='paper',
    text='ADF p = 0.551  |  KPSS p = 0.010<br>→ NON-STATIONARY',
    showarrow=False,
    font=dict(size=10, color=RED, family='Arial'),
    bgcolor='white',
    bordercolor=RED,
    borderwidth=1,
)
fig3.add_annotation(
    x=0.78, y=0.05,
    xref='paper', yref='paper',
    text='ADF p = 0.000  |  KPSS p = 0.100<br>→ STATIONARY',
    showarrow=False,
    font=dict(size=10, color=GREEN, family='Arial'),
    bgcolor='white',
    bordercolor=GREEN,
    borderwidth=1,
)

fig3.update_layout(
    title=dict(
        text=(f'Figure 3.3 — Bond Yield 10Y: Level vs First Difference ({COUNTRY})<br>'
              '<sup>First differencing resolves non-stationarity — '
              'quarterly change in yield enters the model</sup>'),
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    template=TEMPLATE,
    height=380,
    margin=dict(t=100, b=60, l=60, r=40),
)
fig3.update_yaxes(title_text='Yield (%)', row=1, col=1)
fig3.update_yaxes(title_text='Change in Yield (%)', row=1, col=2)
fig3.show()

## **4: Distributional Analysis & Outlier Detection**
___

Once we know the stationarity of the variables, we examine their shapes.

Bank & Eder (2021, Section 10, pp. 34–35) flag the consideration of forward-looking information as one of the biggest challenges in IFRS 9 implementation, citing deRitis, Licari & Ordonez-Sanz (2018) as the key practitioner reference for scenario design.Their framework requires that macro variables behave consistently across historical stress periods before they are used to construct scenarios. This step is the empirical validation of that requirement — each variable must move in the economically expected direction during recession periods, and distributional properties must be understood before any modelling decisions are made.

Fat tails, skewness, and outliers matter here for two reasons. First, variables with extreme observations concentrated in stress periods (GFC, COVID) are carrying exactly the signal you need for IFRS 9 scenario analysis — those observations must be retained, not removed. Second, knowing the distributional shape of each predictor informs model choice downstream: a variable with kurtosis well above 3 will behave differently in a linear regression versus a tree-based model, and that asymmetry needs to be documented.

In [19]:
# 4.1  Distributional summary table
dist_rows = []
for var in FEATURES_STAT + [TARGET]:
    s   = df_model[var].dropna()
    iqr = s.quantile(0.75) - s.quantile(0.25)
    outliers = int(((s < s.quantile(0.25) - 3*iqr) |
                    (s > s.quantile(0.75) + 3*iqr)).sum())
    dist_rows.append({
        'Variable':         var.replace(PREFIX,'').replace('_',' ').title(),
        'N':                len(s),
        'Mean':             round(s.mean(), 3),
        'Median':           round(s.median(), 3),
        'Std':              round(s.std(), 3),
        'Min':              round(s.min(), 3),
        'Max':              round(s.max(), 3),
        'Skewness':         round(s.skew(), 3),
        'Kurtosis':         round(s.kurtosis(), 3),
        'Outliers (3×IQR)': outliers,
    })

dist_df = pd.DataFrame(dist_rows)

def colour_skew(val):
    if abs(val) > 2: return 'background-color: #FADBD8'
    if abs(val) > 1: return 'background-color: #FEF9E7'
    return 'background-color: #D5F5E3'

def colour_kurt(val):
    if abs(val) > 5: return 'background-color: #FADBD8'
    if abs(val) > 3: return 'background-color: #FEF9E7'
    return 'background-color: #D5F5E3'

def colour_outliers(val):
    if val > 5: return 'background-color: #FADBD8'
    if val > 2: return 'background-color: #FEF9E7'
    return 'background-color: #D5F5E3'

display(dist_df.style
        .map(colour_skew,     subset=['Skewness'])
        .map(colour_kurt,     subset=['Kurtosis'])
        .map(colour_outliers, subset=['Outliers (3×IQR)'])
        .format({c: '{:.3f}' for c in
                 ['Mean','Median','Std','Min','Max','Skewness','Kurtosis']})
        .set_caption(f'Table 4.1 — Distributional Summary ({COUNTRY}, 1990–2024)')
        .hide(axis='index'))

Variable,N,Mean,Median,Std,Min,Max,Skewness,Kurtosis,Outliers (3×IQR)
Gdp Yoy Growth,135,2.556,2.673,2.032,-7.399,12.386,-0.730,8.386,3
Unemployment,135,5.711,5.333,1.772,3.533,13.000,1.179,1.445,0
Cpi,135,2.601,2.539,1.536,-1.623,8.636,1.201,3.996,3
Consumer Confidence,135,91.887,94.145,14.133,60.360,118.497,-0.309,-0.652,0
Bond Yield 10Y D1,135,-0.028,0.000,0.461,-1.270,1.010,-0.084,-0.267,0
Credit Qoq Growth,135,1.287,1.247,0.884,-0.938,2.994,-0.312,-0.258,0
Sp500 Qoq Growth,135,2.369,3.146,7.808,-22.558,20.867,-0.692,0.980,0
Log Vix,135,2.906,2.863,0.345,2.252,3.980,0.676,0.141,0
House Price Yoy,135,2.252,3.436,4.630,-12.567,13.026,-0.786,1.153,0
Indprod Yoy,135,1.529,2.447,4.054,-15.282,8.958,-1.545,3.906,2


In [20]:
# ── 4.2  Histogram + KDE panel ────────────────────────────────────────────────
plot_vars = FEATURES_STAT + [TARGET]
ncols = 3
nrows = int(np.ceil(len(plot_vars) / ncols))

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[v.replace(PREFIX,'').replace('_',' ').title()
                    for v in plot_vars],
    vertical_spacing=0.10,
    horizontal_spacing=0.07,
)

for i, var in enumerate(plot_vars):
    row = i // ncols + 1
    col = i %  ncols + 1
    s   = df_model[var].dropna()
    is_target = var == TARGET
    color = RED if is_target else BLUE

    # Histogram
    fig.add_trace(
        go.Histogram(
            x=s,
            nbinsx=30,
            marker_color=color,
            opacity=0.55,
            histnorm='probability density',
            name=var,
            hovertemplate='%{x:.2f}<br>Density: %{y:.3f}<extra></extra>',
            showlegend=False,
        ),
        row=row, col=col,
    )

    # KDE overlay
    kde_x = np.linspace(s.min(), s.max(), 200)
    kde_y = stats.gaussian_kde(s)(kde_x)
    fig.add_trace(
        go.Scatter(
            x=kde_x, y=kde_y,
            mode='lines',
            line=dict(color=NAVY, width=2),
            showlegend=False,
            hovertemplate='KDE: %{y:.3f}<extra></extra>',
        ),
        row=row, col=col,
    )

    # Normal reference curve
    norm_y = stats.norm.pdf(kde_x, s.mean(), s.std())
    fig.add_trace(
        go.Scatter(
            x=kde_x, y=norm_y,
            mode='lines',
            line=dict(color=GREY, width=1.5, dash='dash'),
            showlegend=False,
            hovertemplate='Normal ref: %{y:.3f}<extra></extra>',
        ),
        row=row, col=col,
    )

# Skewness + kurtosis annotation per subplot
    skew_val = s.skew()
    kurt_val = s.kurtosis()
    annot_color = (RED if abs(skew_val) > 2
                   else AMBER if abs(skew_val) > 1
                   else GREEN)

    # Plotly subplot xref: first subplot is 'x', rest are 'x2', 'x3' etc
    x_ref = 'x domain' if i == 0 else f'x{i+1} domain'
    y_ref = 'y domain' if i == 0 else f'y{i+1} domain'

    fig.add_annotation(
        xref=x_ref,
        yref=y_ref,
        x=0.97, y=0.95,
        text=f'skew={skew_val:.2f}<br>kurt={kurt_val:.2f}',
        showarrow=False,
        font=dict(size=8, color=annot_color, family='Arial'),
        bgcolor='white',
        bordercolor=annot_color,
        borderwidth=0.8,
        align='right',
    )

fig.update_layout(
    title=dict(
        text=(f'Figure 4.1 — Histograms + KDE ({COUNTRY}, 1990–2024)  |  '
              f'navy = KDE  |  grey dashed = normal reference  |  '
              f'red = target'),
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    template=TEMPLATE,
    height=nrows * 260,
    margin=dict(t=80, b=40, l=50, r=40),
)
fig.show()

In [21]:
# ── 4.3  Recession stress behaviour table ─────────────────────────────────────
in_recession = pd.Series(False, index=df_model.index)
for start, end, label in RECESSIONS:
    in_recession |= ((df_model.index >= start) &
                     (df_model.index <= end))

stress = pd.DataFrame({
    'During recession':  df_model[FEATURES_STAT].loc[in_recession].median(),
    'Outside recession': df_model[FEATURES_STAT].loc[~in_recession].median(),
})
stress['Difference'] = (stress['During recession']
                        - stress['Outside recession'])

def expected_sign(varname):
    pos_kws = ['vix', 'cpi']
    neg_kws = ['gdp', 'sp500', 'ftse', 'indprod',
               'house_price', 'credit_qoq', 'oil',
               'consumer_confidence', 'bond_yield']
    amb_kws = ['unemployment', 'reer']
    vl = varname.lower()
    for kw in pos_kws:
        if kw in vl: return 'Expect positive'
    for kw in neg_kws:
        if kw in vl: return 'Expect negative'
    for kw in amb_kws:
        if kw in vl: return 'Ambiguous'
    return 'Ambiguous'

stress['Expected'] = [expected_sign(v) for v in stress.index]
stress['Confirmed'] = [
    'Yes' if (
        (stress.loc[v, 'Difference'] > 0
         and 'positive' in stress.loc[v, 'Expected'])
        or
        (stress.loc[v, 'Difference'] < 0
         and 'negative' in stress.loc[v, 'Expected'])
    )
    else ('N/A' if 'Ambiguous' in stress.loc[v, 'Expected']
          else 'Check')
    for v in stress.index
]

def colour_confirmed(val):
    if val == 'Yes':   return 'background-color: #D5F5E3'
    if val == 'Check': return 'background-color: #FADBD8'
    return ''

def colour_diff(val):
    if val < -0.5: return 'background-color: #EBF5FB'
    if val >  0.5: return 'background-color: #FDEBD0'
    return ''

display(stress.style
        .map(colour_confirmed, subset=['Confirmed'])
        .map(colour_diff,      subset=['Difference'])
        .format({
            'During recession':  '{:.3f}',
            'Outside recession': '{:.3f}',
            'Difference':        '{:+.3f}',
        })
        .set_caption(
            f'Table 4.2 — Median Values During vs Outside Recessions '
            f'({COUNTRY})'))

,During recession,Outside recession,Difference,Expected,Confirmed
us_gdp_yoy_growth,0.743,2.797,-2.054,Expect negative,Yes
us_unemployment,5.167,5.433,-0.267,Ambiguous,N/A
us_cpi,3.037,2.516,+0.521,Expect positive,Yes
us_consumer_confidence,80.911,96.046,-15.135,Expect negative,Yes
us_bond_yield_10y_d1,-0.380,0.000,-0.380,Expect negative,Yes
us_credit_qoq_growth,1.138,1.255,-0.117,Expect negative,Yes
us_sp500_qoq_growth,-9.399,3.663,-13.062,Expect negative,Yes
us_log_vix,3.313,2.825,+0.488,Expect positive,Yes
us_house_price_yoy,-6.241,3.448,-9.689,Expect negative,Yes
us_indprod_yoy,-4.662,2.564,-7.226,Expect negative,Yes


**Notes on Recession Stress Behaviour**

**Unemployment (Ambiguous):** Unemployment is a well-documented lagging indicator — it continues rising for several quarters after a recession officially ends. The NBER recession bands mark the trough of the cycle, not the full unemployment response. The GFC unemployment peak of ~10% occurred in late 2009, well after the recession was declared over in June 2009. The median during recession bands therefore understates the true stress response. This does not invalidate the variable — it means the lag structure identified in Step 5 (CCF) will be particularly important for unemployment.

**Bond Yield First Difference (Confirmed):** Central banks cut rates during recessions, so the quarterly change in yield is negative during stress periods (-0.380 vs 0.000 outside recessions). This is the correct and expected behaviour for the first-differenced series.

**REER Diff (Ambiguous):** The direction of exchange rate movement during recessions is theoretically indeterminate — it depends on whether the recession is domestic or global, and on capital flow dynamics. Marked ambiguous intentionally.

In [22]:
# ── 4.4  Outlier flag summary ─────────────────────────────────────────────────
stress_periods = pd.Series(
    ((df_model.index >= '2008-01-01') & (df_model.index <= '2009-12-31')) |
    ((df_model.index >= '2020-01-01') & (df_model.index <= '2020-12-31')),
    index=df_model.index
)

outlier_rows = []
for var in FEATURES_STAT:
    s   = df_model[var].dropna()
    iqr = s.quantile(0.75) - s.quantile(0.25)
    is_outlier = ((s < s.quantile(0.25) - 3*iqr) |
                  (s > s.quantile(0.75) + 3*iqr))

    # Align stress_periods index to match is_outlier index
    stress_aligned = stress_periods.reindex(is_outlier.index, fill_value=False)

    total      = int(is_outlier.sum())
    in_stress  = int((is_outlier & stress_aligned).sum())
    out_stress = total - in_stress
    note = ('Retain — stress signal' if in_stress > 0
            else ('Investigate' if out_stress > 0
                  else 'None'))
    outlier_rows.append({
        'Variable':         var.replace(PREFIX,'').replace('_',' ').title(),
        'Total outliers':   total,
        'In stress period': in_stress,
        'Outside stress':   out_stress,
        'Action':           note,
    })

outlier_df = pd.DataFrame(outlier_rows)

def colour_action(val):
    if val == 'Retain — stress signal': return 'background-color: #FEF9E7'
    if val == 'Investigate':            return 'background-color: #FADBD8'
    return 'background-color: #D5F5E3'

display(outlier_df.style
        .map(colour_action, subset=['Action'])
        .set_caption(
            f'Table 4.3 — Outlier Flag Summary ({COUNTRY})  |  '
            f'Stress-period outliers retained for IFRS 9 scenario analysis')
        .hide(axis='index'))

Variable,Total outliers,In stress period,Outside stress,Action
Gdp Yoy Growth,3,2,1,Retain — stress signal
Unemployment,0,0,0,None
Cpi,3,0,3,Investigate
Consumer Confidence,0,0,0,None
Bond Yield 10Y D1,0,0,0,None
Credit Qoq Growth,0,0,0,None
Sp500 Qoq Growth,0,0,0,None
Log Vix,0,0,0,None
House Price Yoy,0,0,0,None
Indprod Yoy,2,2,0,Retain — stress signal


**Notes on Outlier Flags**

**CPI (3 outliers outside defined stress windows):** The post-COVID inflation surge (2021–2023) produces extreme CPI readings that fall outside the defined GFC (2008–2009) and COVID (2020) stress windows. These are legitimate macro observations reflecting the supply-chain disruption and energy price shock following the pandemic. They are retained — removing them would eliminate exactly the kind of stress signal that IFRS 9 scenario analysis requires.

**Oil YoY (1 outlier outside stress window):** The 2021 base-effect spike (max 188.59% in Table 4.1) falls just outside the defined COVID stress window. This reflects the oil price recovery from near-zero in April 2020 — an artefact of the YoY calculation rather than a genuine demand signal. Retained but noted as a base-effect distortion that may require treatment in the modelling stage.

## **5: Lag Structure & Cross-Correlation Analysis**
___

The cross-correlation function analysis is the most directly relevant EDA step for this project's research questions. The BDO Proposal (Research Question 1) asks which macroeconomic indicators are most influential in predicting default risk, and Research Question 2 asks how macroeconomic forecasts are translated into forward-looking PD estimates. Both questions require knowing not just whether a variable predicts defaults, but at which lag it does so most strongly. The lag structure determines which lagged regressors enter the Stage 2 PD model and how far ahead macro variables must be forecast in Stage 1 to generate meaningful forward-looking estimates.

Djeundje & Crook (2019), cited in Bank & Eder (2021, Section 6.2, pp. 18–19), demonstrate that models where macroeconomic variables' effects on PD change over account duration outperform constant-coefficient approaches. Their work uses explicit lag analysis as the foundation for model specification, which is precisely what this step produces. Krüger, Rösch & Scheule (2018), cited in Bank & Eder (2021, Table 4, p. 41), use quarterly macro lags of 1–4 quarters in a PD framework and provide the empirical benchmark for expected results here.

In [ ]:
# 5.1  Compute CCF: each feature vs delinquency rate
# statsmodels ccf(y, x, nlags=k) returns lags 1..k
# Lag 0 prepended manually as Pearson correlation

MAX_LAG = 8
N       = len(df_model)
CONF    = 1.96 / np.sqrt(N)   # 95% confidence bound

def compute_ccf(y_arr, x_arr, max_lag):
    lag0     = np.corrcoef(y_arr, x_arr)[0, 1]
    lags_1on = ccf(y_arr, x_arr, nlags=max_lag, alpha=None)
    return np.concatenate([[lag0], lags_1on])  # shape (max_lag+1,)

target_arr = df_model[TARGET].values
ccf_results = {}
for var in FEATURES_STAT:
    ccf_results[var] = compute_ccf(
        target_arr,
        df_model[var].values,
        MAX_LAG
    )

print(f'CCF computed for {len(ccf_results)} variables at lags 0–{MAX_LAG}.')
print(f'95% confidence bound: ±{CONF:.3f}')

CCF computed for 12 variables at lags 0–8.
95% confidence bound: ±0.169


In [ ]:
# 5.2  CCF bar chart panel vs target
lags_arr = np.arange(0, MAX_LAG + 1)
ncols    = 3
nrows    = int(np.ceil(len(FEATURES_STAT) / ncols))

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[v.replace(PREFIX,'').replace('_',' ').title()
                    for v in FEATURES_STAT],
    vertical_spacing=0.10,
    horizontal_spacing=0.08,
)

for i, var in enumerate(FEATURES_STAT):
    row  = i // ncols + 1
    col  = i %  ncols + 1
    vals = ccf_results[var]

    # Bar colours: green = positive correlation, red = negative
    bar_colors = [GREEN if v > 0 else RED for v in vals]

    # Black border on bars exceeding confidence bound
    border_colors = ['black' if abs(v) > CONF else 'rgba(0,0,0,0)'
                     for v in vals]
    border_widths = [1.5 if abs(v) > CONF else 0 for v in vals]

    fig.add_trace(
        go.Bar(
            x=lags_arr,
            y=vals,
            marker_color=bar_colors,
            marker_line_color=border_colors,
            marker_line_width=border_widths,
            name=var,
            hovertemplate='Lag %{x}Q<br>CCF = %{y:.3f}<extra></extra>',
            showlegend=False,
        ),
        row=row, col=col,
    )

    # Confidence bands
    fig.add_hline(y= CONF, line_dash='dot',
                  line_color=GREY, line_width=1,
                  row=row, col=col)
    fig.add_hline(y=-CONF, line_dash='dot',
                  line_color=GREY, line_width=1,
                  row=row, col=col)
    fig.add_hline(y=0, line_color='black',
                  line_width=0.5,
                  row=row, col=col)

    # Peak lag annotation
    peak_lag = int(np.argmax(np.abs(vals)))
    peak_val = vals[peak_lag]
    fig.add_annotation(
        xref=f'x{i+1} domain' if i > 0 else 'x domain',
        yref=f'y{i+1} domain' if i > 0 else 'y domain',
        x=0.97, y=0.95,
        text=f'peak lag {peak_lag}Q<br>CCF={peak_val:.2f}',
        showarrow=False,
        font=dict(size=8, color=NAVY, family='Arial'),
        bgcolor='white',
        bordercolor=NAVY,
        borderwidth=0.8,
        align='right',
    )

fig.update_layout(
    title=dict(
        text=(f'Figure 5.1 — CCF: Each Feature vs {TARGET} ({COUNTRY})  |  '
              f'black border = significant at 95%  |  '
              f'green = positive, red = negative  |  '
              f'dotted = ±{CONF:.2f} confidence bound'),
        font=dict(size=12, color=NAVY, family='Arial'),
    ),
    template=TEMPLATE,
    height=nrows * 260,
    margin=dict(t=100, b=40, l=50, r=40),
)
fig.update_xaxes(title_text='Lag (quarters)', tickvals=list(lags_arr))
fig.show()

In [ ]:
# 5.3  Lag summary table — ranked by peak CCF magnitude
lag_rows = []
for var in FEATURES_STAT:
    vals     = ccf_results[var]
    peak_lag = int(np.argmax(np.abs(vals)))
    peak_val = vals[peak_lag]
    lag_rows.append({
        'Variable':        var,
        'Peak CCF':        round(peak_val, 3),
        '|Peak CCF|':      round(abs(peak_val), 3),
        'Peak lag (Q)':    peak_lag,
        'Direction':       'Positive' if peak_val > 0 else 'Negative',
        'Significant 95%': 'Yes' if abs(peak_val) > CONF else 'No',
    })

lag_df = (pd.DataFrame(lag_rows)
          .sort_values('|Peak CCF|', ascending=False)
          .reset_index(drop=True))

def colour_sig(val):
    if val == 'Yes': return 'background-color: #D5F5E3; font-weight: bold'
    return 'background-color: #FADBD8'

def colour_direction(val):
    if val == 'Positive': return 'background-color: #FDEBD0'
    return 'background-color: #EBF5FB'

display(lag_df.style
        .map(colour_sig,       subset=['Significant 95%'])
        .map(colour_direction, subset=['Direction'])
        .set_caption(
            f'Table 5.1 — Lag Summary: Ranked by Peak CCF Magnitude  |  '
            f'Direct answer to Research Question 1 ({COUNTRY})')
        .hide(axis='index'))

Variable,Peak CCF,|Peak CCF|,Peak lag (Q),Direction,Significant 95%
us_house_price_yoy,-0.838000,0.838000,6,Negative,Yes
us_unemployment,0.806000,0.806000,1,Positive,Yes
us_credit_qoq_growth,-0.679000,0.679000,2,Negative,Yes
us_consumer_confidence,-0.474000,0.474000,5,Negative,Yes
us_gdp_yoy_growth,-0.430000,0.430000,4,Negative,Yes
us_log_vix,0.294000,0.294000,5,Positive,Yes
us_indprod_yoy,-0.282000,0.282000,5,Negative,Yes
us_cpi,-0.257000,0.257000,0,Negative,Yes
us_sp500_qoq_growth,-0.161000,0.161000,7,Negative,No
us_bond_yield_10y_d1,-0.137000,0.137000,6,Negative,No


In [ ]:
# 5.4  Build lag_spec — candidate lag per variable
# Adds lagged columns to df_model — originals remain untouched
lag_spec = {}
for _, row in lag_df.iterrows():
    var = row['Variable']
    k   = int(row['Peak lag (Q)'])
    lag_col = var + '_L' + str(k)
    df_model[lag_col] = df_model[var].shift(k)
    lag_spec[var] = {'lag': k, 'col': lag_col}

print('Candidate lag specifications:')
print(f'{"Variable":42s}  {"Lag":>5}  {"New column":40s}  {"Sig":>4}')
print('-' * 100)
for _, row in lag_df.iterrows():
    var  = row['Variable']
    spec = lag_spec[var]
    sig  = row['Significant 95%']
    print(f'{var:42s}  {spec["lag"]:>5}Q  '
          f'{spec["col"]:40s}  {sig:>4}')

print(f'\ndf_model now has {df_model.shape[1]} columns '
      f'({len(lag_spec)} lagged columns added, originals retained)')

Candidate lag specifications:
Variable                                      Lag  New column                                 Sig
----------------------------------------------------------------------------------------------------
us_house_price_yoy                              6Q  us_house_price_yoy_L6                      Yes
us_unemployment                                 1Q  us_unemployment_L1                         Yes
us_credit_qoq_growth                            2Q  us_credit_qoq_growth_L2                    Yes
us_consumer_confidence                          5Q  us_consumer_confidence_L5                  Yes
us_gdp_yoy_growth                               4Q  us_gdp_yoy_growth_L4                       Yes
us_log_vix                                      5Q  us_log_vix_L5                              Yes
us_indprod_yoy                                  5Q  us_indprod_yoy_L5                          Yes
us_cpi                                          0Q  us_cpi_L0                 

In [ ]:
# 5.5  CCF magnitude ranking — interactive bar chart
fig2 = go.Figure()

colors = [GREEN if d == 'Positive' else RED
          for d in lag_df['Direction']]
border = ['black' if s == 'Yes' else 'grey'
          for s in lag_df['Significant 95%']]

fig2.add_trace(go.Bar(
    x=lag_df['|Peak CCF|'],
    y=lag_df['Variable'].str.replace(PREFIX,'').str.replace('_',' ').str.title(),
    orientation='h',
    marker_color=colors,
    marker_line_color=border,
    marker_line_width=1.5,
    text=[f'lag {k}Q  |  CCF={v:+.2f}'
          for k, v in zip(lag_df['Peak lag (Q)'],
                          lag_df['Peak CCF'])],
    textposition='outside',
    hovertemplate='%{y}<br>|Peak CCF| = %{x:.3f}<extra></extra>',
))

fig2.add_vline(
    x=CONF,
    line_dash='dash', line_color=NAVY, line_width=1.5,
    annotation_text=f'95% CI = {CONF:.2f}',
    annotation_font_color=NAVY,
)

fig2.update_layout(
    title=dict(
        text=(f'Figure 5.2 — Variable Importance Ranking by Peak CCF ({COUNTRY})<br>'
              f'<sup>green = positive correlation with delinquency rate  |  '
              f'red = negative  |  black border = significant at 95%</sup>'),
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    xaxis_title='|Peak CCF|',
    xaxis=dict(range=[0, lag_df['|Peak CCF|'].max() + 0.15]),
    template=TEMPLATE,
    height=480,
    margin=dict(t=100, b=40, l=180, r=180),
)
fig2.update_yaxes(autorange='reversed')
fig2.show()

**Top Predictors of US Delinquency Rate (Significant at 95%)**

| Variable | Peak CCF | Lag | Interpretation |
|----------|----------|-----|----------------|
| us_house_price_yoy | -0.838 | 6Q | Strongest predictor. House price declines lead defaults by 18 months — consistent with mortgage default transmission mechanism. |
| us_unemployment | +0.806 | 1Q | Second strongest. Unemployment rise leads defaults by one quarter. Despite being a lagging economic indicator, it is the dominant short-term default signal. |
| us_credit_qoq_growth | -0.679 | 2Q | Credit contraction leads defaults by 2 quarters — bank lending tightening is a strong near-term signal. |
| us_consumer_confidence | -0.474 | 5Q | Confidence collapse leads defaults by over a year — valuable forward-looking signal for IFRS 9 scenario analysis. |
| us_gdp_yoy_growth | -0.430 | 4Q | GDP slowdown leads defaults by 4 quarters — one full year of lead time. |
| us_log_vix | +0.294 | 5Q | Market fear precedes credit stress with a longer lag than expected. |
| us_indprod_yoy | -0.282 | 5Q | Industrial slowdown leads defaults by 5 quarters. |
| us_cpi | -0.257 | 0Q | Contemporaneous relationship only — no forward-looking lead. Limited value for scenario projection. |

**Not Significant at 95% — Deprioritised for Stage 2**
| Variable | Peak CCF | Lag | Note |
|----------|----------|-----|------|
| us_sp500_qoq_growth | -0.161 | 7Q | Quarterly equity returns too noisy for aggregate default prediction. |
| us_bond_yield_10y_d1 | -0.137 | 6Q | Rate changes have limited direct predictive power for aggregate delinquency. |
| us_oil_yoy | -0.132 | 2Q | Weak signal — oil price changes not a primary driver of US commercial bank defaults. |
| us_reer_diff | -0.101 | 1Q | Exchange rate changes have minimal predictive power for domestic delinquency. |

**Implications for Stage 2 Modelling**
The lag structure identified here directly informs the feature construction 
for Stage 2 PD models. Each significant variable enters the model at its 
candidate lag rather than contemporaneously. The four non-significant 
variables will be reviewed in Step 6 multicollinearity assessment before 
a final inclusion decision is made.

Note on Lag Structure for Stage 2 Modelling

The CCF analysis identifies the lag at which each variable has its strongest empirical relationship with the delinquency rate. These candidate lags inform but do not rigidly determine the Stage 2 model specification. Three approaches will be evaluated:

1. **Variable-specific lags** — each predictor enters at its CCF peak lag.
   Maximises predictive signal but creates an asymmetric information set.

2. **Common lag horizon** — all predictors enter at a fixed lag (e.g. 4Q).
   Simpler and more interpretable for regulatory purposes.

3. **Multi-horizon models** — separate models estimated at 1Q, 4Q, and 8Q 
   forecast horizons following Breeden & Crook (2020). Most appropriate for IFRS 9 lifetime ECL estimation.

The final choice will be determined during Stage 2 model development.

### **6:Multicollinearity Assessment (Correlation + VIF + Clustering)** 
___

With approximately 134 quarterly observations and 12 potential predictors, the observation-to-predictor ratio is tight. Multicollinearity inflates standard errors, destabilises coefficient estimates, and makes it impossible to disentangle the individual contribution of correlated predictors — particularly problematic for IFRS 9 scenario analysis where each variable's directional effect on the delinquency rate must be interpretable and economically sensible. Bank & Eder (2021, p. 24) note that a parsimonious parametrisation must be found for reliable estimation in macro-PD discrete-time models. Breeden & Crook (2020), cited in Bank & Eder (2021, Table 4, p. 41), face the identical variable selection problem in a quarterly macro-PD framework.

This step produces two feature sets. The baseline set (VIF < 5) is for non-regularised models such as OLS and logistic regression. The extended set (VIF < 10) is for regularised ML models such as Ridge, Lasso, XGBoost, and MLP where multicollinearity is handled algorithmically. Only the eight significant variables from Step 5 enter this assessment — the four non-significant variables (S&P 500, bond yield d1, oil YoY, REER diff) are excluded from the correlation and VIF analysis but remain in df_model.

**Note on Lagged Variables in Multicollinearity Assessment**

The multicollinearity assessment is performed on each variable at its candidate lag identified in Step 5, not on the contemporaneous values.

This is a deliberate methodological choice. In Step 5 the CCF analysis established that the relationship between each macro variable and the delinquency rate operates at a specific lag — unemployment at lag 1Q, house prices at lag 6Q, GDP growth at lag 4Q, and so on. The Stage 2 PD model will not enter `unemployment(t)` but `unemployment(t-1)`, not `house_price_yoy(t)` but `house_price_yoy(t-6)`.

Checking multicollinearity between the original contemporaneous values would describe correlations that do not reflect the actual model being built. What matters is whether the variables are collinear **as they will actually enter the model** — that is, at their respective lags.

As a concrete example: `us_house_price_yoy` at lag 6Q and `us_unemployment` at lag 1Q are measuring the economy at completely different points in time. Their correlation at those specific lags is the operationally relevant quantity, not their contemporaneous correlation. Running VIF on contemporaneous values would overstate or understate the true multicollinearity problem in the model.

This approach is consistent with the variable selection framework in Breeden & Crook (2020), cited in Bank & Eder (2021, Table 4, p. 41), where predictor collinearity is assessed at the lag specifications used in the final model rather than at contemporaneous values.

In [ ]:
# 6.1  Define feature groups for multicollinearity analysis
SIG_VARS     = lag_df[lag_df['Significant 95%'] == 'Yes']['Variable'].tolist()
NON_SIG_VARS = lag_df[lag_df['Significant 95%'] == 'No']['Variable'].tolist()
EXCL_VARS    = list(FEATURES_EXCLUDED.keys())

# All variables with lag specs (significant + non-significant from Step 5)
ALL_MODEL_VARS  = SIG_VARS + NON_SIG_VARS
ALL_LAGGED_COLS = [lag_spec[v]['col'] for v in ALL_MODEL_VARS
                   if v in lag_spec]

# Significant lagged cols only — used for VIF
SIG_LAGGED_COLS = [lag_spec[v]['col'] for v in SIG_VARS]

# Working matrices
df_mc      = df_model[SIG_LAGGED_COLS].dropna().copy()   # VIF only
df_mc_full = df_model[ALL_LAGGED_COLS].dropna().copy()   # heatmap

print('Variable groups:')
print(f'\n  Significant (Step 5, enter model):        {len(SIG_VARS)}')
for v in SIG_VARS:
    print(f'    {v:42s}  lag {lag_spec[v]["lag"]}Q')

print(f'\n  Non-significant (Step 5, deprioritised):  {len(NON_SIG_VARS)}')
for v in NON_SIG_VARS:
    print(f'    {v:42s}  lag {lag_spec[v]["lag"]}Q')

print(f'\n  Excluded (Step 2, not modelled):          {len(EXCL_VARS)}')
for v, reason in FEATURES_EXCLUDED.items():
    print(f'    {v:42s}  —  {reason}')

print(f'\nMulticollinearity matrix (all modelled vars): {df_mc_full.shape}')
print(f'VIF matrix (significant vars only):           {df_mc.shape}')

Variable groups:

  Significant (Step 5, enter model):        8
    us_house_price_yoy                          lag 6Q
    us_unemployment                             lag 1Q
    us_credit_qoq_growth                        lag 2Q
    us_consumer_confidence                      lag 5Q
    us_gdp_yoy_growth                           lag 4Q
    us_log_vix                                  lag 5Q
    us_indprod_yoy                              lag 5Q
    us_cpi                                      lag 0Q

  Non-significant (Step 5, deprioritised):  4
    us_sp500_qoq_growth                         lag 7Q
    us_bond_yield_10y_d1                        lag 6Q
    us_oil_yoy                                  lag 2Q
    us_reer_diff                                lag 1Q

  Excluded (Step 2, not modelled):          4
    us_real_gdp                                 —  Level I(1) — redundant with gdp_yoy_growth
    us_credit                                   —  Level I(1) — redundant with credit_qo

In [ ]:
# 6.2  Pearson correlation heatmap — all modelled variables
corr_full = df_mc_full.corr()

# Build display labels — mark non-significant with (*)
label_map = {}
for v in ALL_MODEL_VARS:
    col   = lag_spec[v]['col']
    label = col.replace(PREFIX,'').replace('_',' ').title()
    if v in NON_SIG_VARS:
        label = label + ' ⊗'
    label_map[col] = label

short_labels_full = [label_map[c] for c in corr_full.columns]

# Colour scale: RdBu_r with white at zero
fig = go.Figure()
fig.add_trace(go.Heatmap(
    z=corr_full.values,
    x=short_labels_full,
    y=short_labels_full,
    colorscale='RdBu_r',
    zmid=0, zmin=-1, zmax=1,
    text=[[f'{corr_full.values[i,j]:.2f}'
           for j in range(len(corr_full.columns))]
          for i in range(len(corr_full.columns))],
    texttemplate='%{text}',
    textfont=dict(size=9),
    hovertemplate='%{y} vs %{x}<br>r = %{z:.3f}<extra></extra>',
    colorbar=dict(title='Pearson r', thickness=15),
))

# Gold border: high correlation pairs
HIGH_CORR = 0.70
for i in range(len(corr_full.columns)):
    for j in range(len(corr_full.columns)):
        if i != j and abs(corr_full.values[i,j]) > HIGH_CORR:
            fig.add_shape(
                type='rect',
                x0=j-0.5, x1=j+0.5,
                y0=i-0.5, y1=i+0.5,
                line=dict(color='gold', width=2.5),
            )

# Red dashed border: non-significant variable rows and columns
non_sig_cols = [lag_spec[v]['col'] for v in NON_SIG_VARS]
col_list     = list(corr_full.columns)
for idx, col in enumerate(col_list):
    if col in non_sig_cols:
        # Full column band
        fig.add_shape(
            type='rect',
            x0=idx-0.5, x1=idx+0.5,
            y0=-0.5,    y1=len(col_list)-0.5,
            line=dict(color=RED, width=2, dash='dash'),
            fillcolor='rgba(192,57,43,0.05)',
        )
        # Full row band
        fig.add_shape(
            type='rect',
            x0=-0.5,    x1=len(col_list)-0.5,
            y0=idx-0.5, y1=idx+0.5,
            line=dict(color=RED, width=2, dash='dash'),
            fillcolor='rgba(192,57,43,0.05)',
        )

fig.update_layout(
    title=dict(
        text=(
            f'Figure 6.1 — Pearson Correlation Heatmap ({COUNTRY})<br>'
            f'<sup>All modelled variables at candidate lags  |  '
            f'gold border = |r| > {HIGH_CORR}  |  '
            f'red dashed band = non-significant variables (⊗)  |  '
            f'Step 5 excluded variables not shown</sup>'
        ),
        font=dict(size=12, color=NAVY, family='Arial'),
    ),
    template=TEMPLATE,
    height=700,
    width=820,
    margin=dict(t=110, b=40, l=40, r=40),
)
fig.show()

# Report high correlation pairs
high_pairs = []
cols = list(corr_full.columns)
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        r = corr_full.values[i,j]
        if abs(r) > HIGH_CORR:
            sig_a = '(sig)'   if cols[i] in SIG_LAGGED_COLS else '(non-sig)'
            sig_b = '(sig)'   if cols[j] in SIG_LAGGED_COLS else '(non-sig)'
            high_pairs.append((cols[i], sig_a, cols[j], sig_b, round(r,3)))

print(f'\nHigh-correlation pairs (|r| > {HIGH_CORR}): {len(high_pairs)}')
for a, sa, b, sb, r in sorted(high_pairs,
                                key=lambda x: abs(x[4]), reverse=True):
    print(f'  {a} {sa}  <->  {b} {sb}  r = {r:+.3f}')


High-correlation pairs (|r| > 0.7): 1
  us_house_price_yoy_L6 (sig)  <->  us_unemployment_L1 (sig)  r = -0.728


In [ ]:
# 6.3  Variance Inflation Factors — all modelled variables
X_vif = sm.add_constant(df_mc_full.copy())
vif_data = pd.DataFrame({
    'Variable': df_mc_full.columns,
    'VIF':      [variance_inflation_factor(X_vif.values, i+1)
                 for i in range(df_mc_full.shape[1])],
})
vif_data['VIF'] = vif_data['VIF'].round(2)
vif_data['Short name'] = (vif_data['Variable']
                           .str.replace(PREFIX,'')
                           .str.replace('_',' ')
                           .str.title())

# Tag each variable as significant or non-significant
non_sig_cols = [lag_spec[v]['col'] for v in NON_SIG_VARS]
vif_data['Group'] = vif_data['Variable'].apply(
    lambda v: 'Non-significant ⊗' if v in non_sig_cols else 'Significant'
)
vif_data['Display name'] = vif_data.apply(
    lambda r: r['Short name'] + ' ⊗' if r['Group'] == 'Non-significant ⊗'
    else r['Short name'], axis=1
)
vif_data['Assessment'] = vif_data['VIF'].apply(
    lambda v:
    'Acceptable  (VIF < 5)'          if v < 5  else
    'Moderate  (VIF 5–10)'           if v < 10 else
    'High — regularise  (VIF > 10)'
)
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)

def colour_vif(val):
    if val < 5:   return 'background-color: #D5F5E3'
    if val < 10:  return 'background-color: #FEF9E7'
    return 'background-color: #FADBD8; font-weight: bold'

def colour_group(val):
    if 'Non' in val: return 'background-color: #FADBD8; color: #7F8C8D'
    return 'background-color: #EBF5FB'

display(vif_data[['Display name','Group','VIF','Assessment']].style
        .map(colour_vif,   subset=['VIF'])
        .map(colour_group, subset=['Group'])
        .set_caption(
            f'Table 6.1 — Variance Inflation Factors ({COUNTRY})  |  '
            f'All modelled variables  |  ⊗ = non-significant in Step 5')
        .hide(axis='index'))

# VIF bar chart
bar_colors = []
for _, row in vif_data.iterrows():
    if row['Group'] == 'Non-significant ⊗':
        bar_colors.append(GREY)
    elif row['VIF'] < 5:
        bar_colors.append(GREEN)
    elif row['VIF'] < 10:
        bar_colors.append(AMBER)
    else:
        bar_colors.append(RED)

fig2 = go.Figure(go.Bar(
    x=vif_data['VIF'],
    y=vif_data['Display name'],
    orientation='h',
    marker_color=bar_colors,
    marker_line_color=NAVY,
    marker_line_width=0.8,
    text=vif_data['VIF'],
    textposition='outside',
    hovertemplate='%{y}<br>VIF = %{x:.2f}<extra></extra>',
))
fig2.add_vline(x=5,  line_dash='dash', line_color=AMBER,
               line_width=1.5,
               annotation_text='VIF = 5',
               annotation_font_color=AMBER)
fig2.add_vline(x=10, line_dash='dash', line_color=RED,
               line_width=1.5,
               annotation_text='VIF = 10',
               annotation_font_color=RED)
fig2.update_layout(
    title=dict(
        text=(f'Figure 6.2 — Variance Inflation Factors ({COUNTRY})<br>'
              f'<sup>All modelled variables at candidate lags  |  '
              f'green = VIF < 5  |  amber = 5–10  |  red = > 10  |  '
              f'grey = non-significant (⊗)</sup>'),
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    xaxis_title='VIF',
    xaxis=dict(range=[0, vif_data['VIF'].max() + 3]),
    template=TEMPLATE,
    height=500,
    margin=dict(t=100, b=40, l=220, r=120),
)
fig2.update_yaxes(autorange='reversed')
fig2.show()

Display name,Group,VIF,Assessment
Credit Qoq Growth L2,Significant,3.200000,Acceptable (VIF < 5)
House Price Yoy L6,Significant,2.890000,Acceptable (VIF < 5)
Indprod Yoy L5,Significant,2.570000,Acceptable (VIF < 5)
Unemployment L1,Significant,2.550000,Acceptable (VIF < 5)
Gdp Yoy Growth L4,Significant,2.440000,Acceptable (VIF < 5)
Consumer Confidence L5,Significant,2.280000,Acceptable (VIF < 5)
Cpi L0,Significant,2.030000,Acceptable (VIF < 5)
Oil Yoy L2 ⊗,Non-significant ⊗,1.670000,Acceptable (VIF < 5)
Sp500 Qoq Growth L7 ⊗,Non-significant ⊗,1.330000,Acceptable (VIF < 5)
Log Vix L5,Significant,1.220000,Acceptable (VIF < 5)


In [ ]:
# 6.4  Hierarchical clustering dendrogram — all modelled variables 
dist_matrix = 1 - np.abs(corr_full.values)
np.fill_diagonal(dist_matrix, 0)
condensed = squareform(np.clip(dist_matrix, 0, None))
Z         = linkage(condensed, method='ward')

CUT = 0.45
cluster_ids = fcluster(Z, t=CUT, criterion='distance')
n_clusters  = cluster_ids.max()

# Labels with ⊗ for non-significant
all_short_labels = [label_map[c] for c in corr_full.columns]

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram as scipy_dendrogram

# Custom leaf colours: grey for non-significant
def leaf_colour_func(id):
    col = list(corr_full.columns)[id]
    return GREY if col in non_sig_cols else NAVY

fig3, ax = plt.subplots(figsize=(14, 5))
ddata = scipy_dendrogram(
    Z,
    labels=all_short_labels,
    ax=ax,
    leaf_rotation=45,
    leaf_font_size=9,
    color_threshold=CUT,
)

# Grey out non-significant leaf labels
for lbl in ax.get_xticklabels():
    if '⊗' in lbl.get_text():
        lbl.set_color(GREY)
        lbl.set_fontstyle('italic')

ax.axhline(CUT, color=RED, lw=1.5, ls='--',
           label=f'Cut-off (distance = {CUT})')
ax.set_title(
    f'Figure 6.3 — Hierarchical Clustering Dendrogram ({COUNTRY})\n'
    f'All modelled variables at candidate lags  |  '
    f'Ward linkage  |  distance = 1 – |r|  |  '
    f'grey italic ⊗ = non-significant (Step 5)',
    fontsize=10, color=NAVY)
ax.set_ylabel('Distance (1 – |r|)')
ax.set_facecolor('#F8F9FA')
fig3.patch.set_facecolor('white')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f'\nClusters at distance threshold {CUT}: {n_clusters}')
for c in range(1, n_clusters + 1):
    members = [all_short_labels[i]
               for i, cl in enumerate(cluster_ids) if cl == c]
    print(f'  Cluster {c}: {members}')


Clusters at distance threshold 0.45: 9
  Cluster 1: ['House Price Yoy L6', 'Unemployment L1']
  Cluster 2: ['Credit Qoq Growth L2', 'Consumer Confidence L5']
  Cluster 3: ['Gdp Yoy Growth L4', 'Indprod Yoy L5']
  Cluster 4: ['Sp500 Qoq Growth L7 ⊗']
  Cluster 5: ['Log Vix L5']
  Cluster 6: ['Cpi L0']
  Cluster 7: ['Oil Yoy L2 ⊗']
  Cluster 8: ['Bond Yield 10Y D1 L6 ⊗']
  Cluster 9: ['Reer Diff L1 ⊗']


In [ ]:
# 6.5  Final feature sets
FEATURES_BASELINE = vif_data[
    (vif_data['VIF'] < 5) &
    (vif_data['Group'] == 'Significant')
]['Variable'].tolist()

FEATURES_EXTENDED = vif_data[
    (vif_data['VIF'] < 10) &
    (vif_data['Group'] == 'Significant')
]['Variable'].tolist()

FEATURES_HIGH_VIF = vif_data[
    (vif_data['VIF'] >= 10) &
    (vif_data['Group'] == 'Significant')
]['Variable'].tolist()

print('=' * 65)
print('FINAL FEATURE SET RECOMMENDATIONS')
print('=' * 65)

print(f'\nBaseline ({len(FEATURES_BASELINE)} vars) — OLS / logit (non-regularised):')
for v in FEATURES_BASELINE:
    vif_val = vif_data.loc[vif_data['Variable']==v, 'VIF'].values[0]
    print(f'  {v:50s}  VIF = {vif_val:.2f}')

print(f'\nExtended ({len(FEATURES_EXTENDED)} vars) — Ridge / Lasso / XGBoost / MLP:')
for v in FEATURES_EXTENDED:
    vif_val = vif_data.loc[vif_data['Variable']==v, 'VIF'].values[0]
    print(f'  {v:50s}  VIF = {vif_val:.2f}')

if FEATURES_HIGH_VIF:
    print(f'\nHigh VIF — regularised models only (VIF ≥ 10):')
    for v in FEATURES_HIGH_VIF:
        vif_val = vif_data.loc[vif_data['Variable']==v, 'VIF'].values[0]
        print(f'  {v:50s}  VIF = {vif_val:.2f}')

print(f'\nNon-significant ⊗ (Step 5) — available but deprioritised:')
for v in NON_SIG_VARS:
    vif_val = vif_data.loc[
        vif_data['Variable']==lag_spec[v]['col'], 'VIF'].values[0]
    print(f'  {lag_spec[v]["col"]:50s}  VIF = {vif_val:.2f}')

print(f'\nExcluded (Step 2) — remain in df_model, not used in modelling:')
for v, reason in FEATURES_EXCLUDED.items():
    print(f'  {v:50s}  —  {reason}')

# Save final analytical dataset
output_cols = (FEATURES_EXTENDED +
               [lag_spec[v]['col'] for v in NON_SIG_VARS
                if lag_spec[v]['col'] in df_model.columns] +
               [TARGET])
df_final = df_model[[c for c in output_cols
                      if c in df_model.columns]].dropna().copy()
df_final.to_csv(BASE / 'EDA_analytical_dataset.csv')

print(f'\nSaved: EDA_analytical_dataset.csv')
print(f'Shape: {df_final.shape}')
print(f'Date range: {df_final.index.min().date()} '
      f'to {df_final.index.max().date()}')

FINAL FEATURE SET RECOMMENDATIONS

Baseline (8 vars) — OLS / logit (non-regularised):
  us_credit_qoq_growth_L2                             VIF = 3.20
  us_house_price_yoy_L6                               VIF = 2.89
  us_indprod_yoy_L5                                   VIF = 2.57
  us_unemployment_L1                                  VIF = 2.55
  us_gdp_yoy_growth_L4                                VIF = 2.44
  us_consumer_confidence_L5                           VIF = 2.28
  us_cpi_L0                                           VIF = 2.03
  us_log_vix_L5                                       VIF = 1.22

Extended (8 vars) — Ridge / Lasso / XGBoost / MLP:
  us_credit_qoq_growth_L2                             VIF = 3.20
  us_house_price_yoy_L6                               VIF = 2.89
  us_indprod_yoy_L5                                   VIF = 2.57
  us_unemployment_L1                                  VIF = 2.55
  us_gdp_yoy_growth_L4                                VIF = 2.44
  us_consumer_con

## 7: ACF/PACF Analysis
___

The autocorrelation function (ACF) and partial autocorrelation function (PACF) analysis serves a dual purpose in this project. First, it directly informs the ARIMA order selection for Stage 1 macro forecasting — the ACF identifies the moving average order q while the PACF identifies the autoregressive order p. This is not optional: the BDO Proposal (Research Question 6) explicitly asks which models benchmarked to ARIMA and ARMA are best suited for forecasting macroeconomic indicators, and you cannot benchmark against ARIMA without first specifying it correctly. Second, the autocorrelation structure of the delinquency rate itself informs whether the target variable has memory — a highly persistent target with slow ACF decay suggests that past default rates carry predictive information beyond the macro variables alone, which has direct implications for model specification in Stage 2.

In [33]:
# ── 7.1  ACF/PACF function ────────────────────────────────────────────────────
from statsmodels.tsa.stattools import acf, pacf

def compute_acf_pacf(series, nlags=12):
    """
    Compute ACF and PACF with 95% confidence bounds.
    Returns acf values, pacf values, and confidence bound.
    """
    s    = series.dropna()
    n    = len(s)
    conf = 1.96 / np.sqrt(n)

    acf_vals  = acf(s,  nlags=nlags, fft=True,  alpha=None)
    pacf_vals = pacf(s, nlags=nlags, method='ywm', alpha=None)

    return acf_vals, pacf_vals, conf

print('ACF/PACF functions defined.')
print(f'Max lags: 12 quarters (3 years)')
print(f'Confidence bound at n=134: ±{1.96/np.sqrt(134):.3f}')

ACF/PACF functions defined.
Max lags: 12 quarters (3 years)
Confidence bound at n=134: ±0.169


In [ ]:
# 7.2  ACF/PACF panel — target variable
NLAGS = 12
s_target = df_model[TARGET].dropna()
n_target = len(s_target)
conf     = 1.96 / np.sqrt(n_target)

acf_vals_t, pacf_vals_t, _ = compute_acf_pacf(s_target, NLAGS)
lags_arr = np.arange(0, NLAGS + 1)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'ACF — {TARGET}<br><sup>Identifies MA(q) order — '
        f'lag where ACF cuts off</sup>',
        f'PACF — {TARGET}<br><sup>Identifies AR(p) order — '
        f'lag where PACF cuts off</sup>',
    ],
    horizontal_spacing=0.12,
)

# ACF
fig.add_trace(
    go.Bar(
        x=lags_arr,
        y=acf_vals_t,
        marker_color=[NAVY if i == 0 else
                      (BLUE if abs(v) > conf else LBLUE)
                      for i, v in enumerate(acf_vals_t)],
        marker_line_color='white',
        marker_line_width=0.5,
        name='ACF',
        hovertemplate='Lag %{x}<br>ACF = %{y:.3f}<extra></extra>',
        showlegend=False,
    ),
    row=1, col=1,
)
fig.add_hline(y= conf, line_dash='dot',
              line_color=RED, line_width=1.5,
              row=1, col=1)
fig.add_hline(y=-conf, line_dash='dot',
              line_color=RED, line_width=1.5,
              row=1, col=1)
fig.add_hline(y=0, line_color='black',
              line_width=0.5, row=1, col=1)

# PACF
fig.add_trace(
    go.Bar(
        x=lags_arr,
        y=pacf_vals_t,
        marker_color=[NAVY if i == 0 else
                      (BLUE if abs(v) > conf else LBLUE)
                      for i, v in enumerate(pacf_vals_t)],
        marker_line_color='white',
        marker_line_width=0.5,
        name='PACF',
        hovertemplate='Lag %{x}<br>PACF = %{y:.3f}<extra></extra>',
        showlegend=False,
    ),
    row=1, col=2,
)
fig.add_hline(y= conf, line_dash='dot',
              line_color=RED, line_width=1.5,
              row=1, col=2)
fig.add_hline(y=-conf, line_dash='dot',
              line_color=RED, line_width=1.5,
              row=1, col=2)
fig.add_hline(y=0, line_color='black',
              line_width=0.5, row=1, col=2)

fig.update_layout(
    title=dict(
        text=(f'Figure 7.1 — ACF & PACF: Target Variable ({TARGET}, {COUNTRY})<br>'
              f'<sup>Dark blue = lag 0  |  blue = significant  |  '
              f'light blue = within bounds  |  '
              f'red dotted = ±{conf:.2f} 95% CI</sup>'),
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    template=TEMPLATE,
    height=400,
    margin=dict(t=120, b=40, l=60, r=40),
)
fig.update_xaxes(title_text='Lag (quarters)', tickvals=list(lags_arr))
fig.show()

# Print ACF/PACF values for target
print(f'ACF values — {TARGET}:')
for i, v in enumerate(acf_vals_t):
    sig = '← significant' if abs(v) > conf and i > 0 else ''
    print(f'  Lag {i:2d}: {v:+.3f}  {sig}')
print(f'\nPACF values — {TARGET}:')
for i, v in enumerate(pacf_vals_t):
    sig = '← significant' if abs(v) > conf and i > 0 else ''
    print(f'  Lag {i:2d}: {v:+.3f}  {sig}')

ACF values — delinquency_rate:
  Lag  0: +1.000  
  Lag  1: +0.972  ← significant
  Lag  2: +0.929  ← significant
  Lag  3: +0.873  ← significant
  Lag  4: +0.807  ← significant
  Lag  5: +0.732  ← significant
  Lag  6: +0.652  ← significant
  Lag  7: +0.571  ← significant
  Lag  8: +0.491  ← significant
  Lag  9: +0.412  ← significant
  Lag 10: +0.336  ← significant
  Lag 11: +0.265  ← significant
  Lag 12: +0.198  ← significant

PACF values — delinquency_rate:
  Lag  0: +1.000  
  Lag  1: +0.972  ← significant
  Lag  2: -0.277  ← significant
  Lag  3: -0.208  ← significant
  Lag  4: -0.129  
  Lag  5: -0.115  
  Lag  6: -0.056  
  Lag  7: -0.004  
  Lag  8: -0.005  
  Lag  9: -0.016  
  Lag 10: -0.021  
  Lag 11: +0.020  
  Lag 12: -0.051  


In [ ]:
# 7.3  ACF/PACF panel — all significant predictors
ncols = 2
nrows = len(SIG_VARS)

fig2 = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[
        item for v in SIG_VARS
        for item in [
            v.replace(PREFIX,'').replace('_',' ').title() + ' — ACF',
            v.replace(PREFIX,'').replace('_',' ').title() + ' — PACF',
        ]
    ],
    vertical_spacing=0.06,
    horizontal_spacing=0.10,
)

for i, var in enumerate(SIG_VARS):
    row = i + 1
    s   = df_model[var].dropna()
    n   = len(s)
    c   = 1.96 / np.sqrt(n)

    acf_v, pacf_v, _ = compute_acf_pacf(s, NLAGS)

    # ACF
    fig2.add_trace(
        go.Bar(
            x=lags_arr,
            y=acf_v,
            marker_color=[NAVY if j == 0 else
                          (BLUE if abs(v) > c else LBLUE)
                          for j, v in enumerate(acf_v)],
            marker_line_color='white',
            marker_line_width=0.5,
            showlegend=False,
            hovertemplate='Lag %{x}<br>ACF = %{y:.3f}<extra></extra>',
        ),
        row=row, col=1,
    )
    fig2.add_hline(y= c, line_dash='dot',
                   line_color=RED, line_width=1,
                   row=row, col=1)
    fig2.add_hline(y=-c, line_dash='dot',
                   line_color=RED, line_width=1,
                   row=row, col=1)
    fig2.add_hline(y=0, line_color='black',
                   line_width=0.5, row=row, col=1)

    # PACF
    fig2.add_trace(
        go.Bar(
            x=lags_arr,
            y=pacf_v,
            marker_color=[NAVY if j == 0 else
                          (AMBER if abs(v) > c else '#FAD7A0')
                          for j, v in enumerate(pacf_v)],
            marker_line_color='white',
            marker_line_width=0.5,
            showlegend=False,
            hovertemplate='Lag %{x}<br>PACF = %{y:.3f}<extra></extra>',
        ),
        row=row, col=2,
    )
    fig2.add_hline(y= c, line_dash='dot',
                   line_color=RED, line_width=1,
                   row=row, col=2)
    fig2.add_hline(y=-c, line_dash='dot',
                   line_color=RED, line_width=1,
                   row=row, col=2)
    fig2.add_hline(y=0, line_color='black',
                   line_width=0.5, row=row, col=2)

fig2.update_layout(
    title=dict(
        text=(f'Figure 7.2 — ACF (blue) & PACF (amber): '
              f'Significant Predictors ({COUNTRY})<br>'
              f'<sup>Dark = lag 0  |  coloured = significant  |  '
              f'light = within bounds  |  '
              f'red dotted = 95% CI</sup>'),
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    template=TEMPLATE,
    height=nrows * 200,
    margin=dict(t=100, b=40, l=60, r=40),
)
fig2.update_xaxes(tickvals=list(lags_arr))
fig2.show()

In [ ]:
# 7.4  ARIMA order suggestion table
def suggest_arima_order(series, name, nlags=12):
    """
    Read ACF and PACF to suggest ARIMA order search space.
    p range: from first to last significant PACF lag
    q range: from first to last significant ACF lag
    Pattern: determined by relative decay of ACF vs PACF
    """
    s    = series.dropna()
    n    = len(s)
    conf = 1.96 / np.sqrt(n)

    acf_v  = acf(s,  nlags=nlags, fft=True,   alpha=None)
    pacf_v = pacf(s, nlags=nlags, method='ywm', alpha=None)

    # Significant lags excluding lag 0
    acf_sig  = [i for i in range(1, nlags+1) if abs(acf_v[i])  > conf]
    pacf_sig = [i for i in range(1, nlags+1) if abs(pacf_v[i]) > conf]

    p_min = pacf_sig[0]  if pacf_sig else 0
    p_max = pacf_sig[-1] if pacf_sig else 0
    q_min = acf_sig[0]   if acf_sig  else 0
    q_max = acf_sig[-1]  if acf_sig  else 0

    # Pattern classification
    acf_slow  = len(acf_sig)  > 4
    pacf_slow = len(pacf_sig) > 4

    if acf_slow and not pacf_slow:
        pattern   = 'AR process — PACF cuts off'
        suggested = (f'AR({p_min}) — test up to AR({p_max})'
                     if p_min != p_max
                     else f'AR({p_min})')
    elif pacf_slow and not acf_slow:
        pattern   = 'MA process — ACF cuts off'
        suggested = (f'MA({q_min}) — test up to MA({q_max})'
                     if q_min != q_max
                     else f'MA({q_min})')
    elif acf_slow and pacf_slow:
        pattern   = 'ARMA process — both decay gradually'
        suggested = (f'ARMA({p_min},{q_min}) — '
                     f'test up to ARMA({p_max},{q_max})')
    else:
        pattern   = 'White noise / short memory'
        suggested = 'AR(1) or white noise — minimal lag structure'

    return {
        'Variable':       name,
        'Sig ACF lags':   len(acf_sig),
        'Sig PACF lags':  len(pacf_sig),
        'p range (AR)':   f'{p_min}–{p_max}' if p_min != p_max else str(p_min),
        'q range (MA)':   f'{q_min}–{q_max}' if q_min != q_max else str(q_min),
        'Suggested spec': suggested,
        'Pattern':        pattern,
    }

# Run for target and all significant predictors 
arima_rows = []

# Target first
arima_rows.append(
    suggest_arima_order(df_model[TARGET], TARGET, nlags=12))

# All significant predictors
for var in SIG_VARS:
    arima_rows.append(
        suggest_arima_order(
            df_model[var],
            var.replace(PREFIX,'').replace('_',' ').title(),
            nlags=12))

arima_df = pd.DataFrame(arima_rows)

# Styling
def colour_pattern(val):
    if 'AR process'   in val: return 'background-color: #EBF5FB'
    if 'MA process'   in val: return 'background-color: #FEF9E7'
    if 'ARMA process' in val: return 'background-color: #FDEBD0'
    return 'background-color: #D5F5E3'

def colour_sig_lags(val):
    if val >= 10: return 'background-color: #FADBD8'
    if val >= 5:  return 'background-color: #FEF9E7'
    return 'background-color: #D5F5E3'

display(arima_df.style
        .map(colour_pattern,   subset=['Pattern'])
        .map(colour_sig_lags,  subset=['Sig ACF lags', 'Sig PACF lags'])
        .set_caption(
            f'Table 7.1 — Suggested ARIMA Orders ({COUNTRY})  |  '
            f'p from PACF cutoff, q from ACF cutoff  |  '
            f'Direct input to Stage 1 model specification')
        .hide(axis='index'))

# Print search spaces
print('\nStage 1 ARIMA grid search spaces:')
print(f'{"Variable":35s}  {"Suggested spec":45s}  Pattern')
print('-' * 110)
for _, row in arima_df.iterrows():
    print(f'{row["Variable"]:35s}  '
          f'{row["Suggested spec"]:45s}  '
          f'{row["Pattern"]}')

Variable,Sig ACF lags,Sig PACF lags,p range (AR),q range (MA),Suggested spec,Pattern
delinquency_rate,12,3,1–3,1–12,AR(1) — test up to AR(3),AR process — PACF cuts off
House Price Yoy,12,3,1–5,1–12,AR(1) — test up to AR(5),AR process — PACF cuts off
Unemployment,9,1,1,1–9,AR(1),AR process — PACF cuts off
Credit Qoq Growth,10,4,1–9,1–10,AR(1) — test up to AR(9),AR process — PACF cuts off
Consumer Confidence,12,1,1,1–12,AR(1),AR process — PACF cuts off
Gdp Yoy Growth,3,4,1–9,1–3,AR(1) or white noise — minimal lag structure,White noise / short memory
Log Vix,6,2,1–2,1–6,AR(1) — test up to AR(2),AR process — PACF cuts off
Indprod Yoy,3,6,1–10,1–3,MA(1) — test up to MA(3),MA process — ACF cuts off
Cpi,5,5,1–9,1–5,"ARMA(1,1) — test up to ARMA(9,5)",ARMA process — both decay gradually



Stage 1 ARIMA grid search spaces:
Variable                             Suggested spec                                 Pattern
--------------------------------------------------------------------------------------------------------------
delinquency_rate                     AR(1) — test up to AR(3)                       AR process — PACF cuts off
House Price Yoy                      AR(1) — test up to AR(5)                       AR process — PACF cuts off
Unemployment                         AR(1)                                          AR process — PACF cuts off
Credit Qoq Growth                    AR(1) — test up to AR(9)                       AR process — PACF cuts off
Consumer Confidence                  AR(1)                                          AR process — PACF cuts off
Gdp Yoy Growth                       AR(1) or white noise — minimal lag structure   White noise / short memory
Log Vix                              AR(1) — test up to AR(2)                       AR process —

**ACF/PACF Key Findings**

**Delinquency rate:** Highly persistent AR process — PACF cuts off after lag 3, ACF decays slowly across all 12 lags. Starting specification for Stage 2 target modelling: AR(1), search up to AR(3).

**Dominant pattern:** Almost all significant predictors follow an AR process where the PACF cuts off and the ACF decays slowly. This means autoregressive models (AR, ARIMA) are appropriate for Stage 1 forecasting of each macro variable independently.

**GDP YoY growth:** White noise / short memory — only 3 significant ACF lags. The least persistent variable in the set. Simple AR(1) or ARMA(1,1) sufficient for Stage 1 forecasting.

**CPI:** ARMA process — both ACF and PACF decay gradually. The post-COVID inflation surge (2021–2023) is likely contributing to this mixed pattern. ARMA(1,1) starting specification, test up to ARMA(9,5).

**Industrial production YoY:** MA process — ACF cuts off while PACF decays slowly. Unusual among the predictor set. MA(1) starting 
specification for Stage 1.

**Implication for Stage 1:** The p and q ranges in Table 7.1 define the grid search space for ARIMA order selection. AIC/BIC will select the optimal order within each range. The minimum specification (p_min, q_min) is the starting point; the maximum (p_max, q_max) is the upper bound for the grid search.